# 뉴스 처리: 수집 마감 뒤 요약 재료 만들기 (프롬프트 입력까지)

`news_collect.ipynb`가 24시간 쌓는 기사를 읽어, 수집 마감마다 밤사이 프롬프트 직전까지 처리한다. 정한 것과 근거는 `news_search.md`(1장 한눈에, 3장 처리, 4장 노트북과 파일), 프롬프트 쪽 요구사항은 루트의 `nightly_prompt_requirements.md`.

**사용자가 고르는 두 시각**(1절, 한국 시간, 30분 단위)

| 시각 | 고를 수 있는 값 | 지금 값 | 쓰는 곳 |
|---|---|---|---|
| 표시 시각(`SUMMARY_TIME`) | 06:00~08:00 | 08:00 | 요약을 보여 줄 시각. 수집 마감은 그 30분 전(07:30)이고, 이 마감에 한 번 처리한다 |
| 취침 시각(`BEDTIME`) | 22:00~02:00 | 00:00 | 이 시각 뒤에 발행된 기사만 요약에 넣는다. 그 전(한국 낮·저녁)에 나온 기사는 이전 소식 확인에만 쓴다 |

**흐름**

```
수집 마감 표시 파일(cutoff_0800.json)이 생기면(10절 대기 칸이 30초마다 확인)
  → 보고 날짜 폴더의 기사를 발행 시각으로 나누기
      · 낮·저녁 기사(전날 수집 마감 ~ 취침 시각): 제목이 비슷한 기사끼리 묶어 비교용 낮·저녁 묶음으로 지난 묶음 기록에 등록(요약에 넣지 않음)
      · 요약에 넣을 밤 기사(취침 시각 뒤 발행, 수집 마감 전에 받음): 아래로
  → 묶기(제목 임베딩 large, 평균 유사도 0.75) → 묶음 중요도(매체 수 → 기사 수)
  → 이전 소식 확인(지난 묶음 기록과 비교: 처음 나온 소식/이전 소식의 후속, 미국 동부 날짜로 2일 전부터)
  → 종목별 대표 기사 → 종목마다 10개 묶음
  → 묶음별 본문 받기(원문 주소 복원 → 원문 페이지 → 본문 추출 → 거르기 2·3단계)
  → 프롬프트 입력(그래프 연결 근거 포함, prompt_inputs.jsonl) → 밤사이 프롬프트(팀원)
```

**절**

| 절 | 내용 |
|---|---|
| 1~2 | 설정(경로, 표시 시각·취침 시각)과 작은 도구(보고 날짜, 보고 날짜의 밤 시각, 쌓인 파일 읽기, 알림) |
| 3 | 검색어 표(종목, 검색어, 종류, 그래프 노드) |
| 4 | 묶기: 임베딩(캐시), 묶기·중요도, 종목별 대표·후보 제목, 묶음별 본문 기사 |
| 5 | 본문 받기 |
| 6 | 그래프 연결 근거 |
| 7 | 이전 소식 확인 |
| 8 | 프롬프트 입력 |
| 9 | 수집 마감 뒤 처리(`summary_run`) |
| 10 | **대기하며 돌리기**: `WATCH_RUN = True`로 켜 두면 수집 마감 표시가 생길 때마다 처리 |
| 11 | 한 보고 날짜 골라 돌리기(다시 돌려 보기) |
| 12 | **결과 보기**: 보고 날짜 하나의 수집·처리 결과를 `report.md`로 정리(요청 없음) |
| 13 | 확인: 3차 시험 기사로 묶기·대표 고르기(요청 없음) |

**읽는 파일**(`news_collect.ipynb`가 쌓음): `source/nightly/<보고 날짜>/items.csv`·`hits.csv`·`cutoff_<표시 시각>.json`. **만드는 파일**: `source/nightly/<보고 날짜>/summary_<표시 시각>/`(요약 묶음, 비교용 낮·저녁 묶음, 고른 묶음, 본문, 프롬프트 입력, 걸린 시간), 캐시 `source/embed_cache/`(임베딩)·`source/bodies/`(원문), `source/stories/`(지난 묶음 기록: 이전 소식 확인에 쓰는 묶음 목록과 중심 벡터).

**한눈에**

| | |
|---|---|
| 외부 요청 | OpenAI 임베딩(하룻밤 $0.01~0.03), 원문 주소 복원·원문 페이지(마감마다 백여 건, 무료). `WATCH_RUN`·`PROCESS_RUN`·`EMBED_RUN`이 True일 때만 |
| 아직 임시값 | 묶기·이전 소식 확인 기준 0.75, 종목당 10개(`news_search.md` 결정할 것 2·5) |

## 1. 설정

In [1]:
# ── 불러오기 ──
import datetime as dt
import gzip
import hashlib
import json
import re
import time
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 220)
EMBED_RUN = False    # True여야 OpenAI 임베딩 요청을 보낸다(캐시에 없는 제목만). 대기·골라 돌리기 칸은 임베딩을 켜고 돈다

In [ ]:
# ── 처리 설정: 경로(수집 노트북과 같은 폴더를 읽는다), 표시 시각·취침 시각 ──
ROOT = Path.cwd()                               # news_search 폴더(노트북을 이 폴더에서 연다)
PROJ = ROOT.parent                              # 프로젝트 루트(.env, graph_db가 있는 곳)
NIGHTLY_DIR = ROOT / 'source' / 'nightly'       # news_collect.ipynb가 보고 날짜마다 쌓는 폴더
KST = ZoneInfo('Asia/Seoul')
ET = ZoneInfo('America/New_York')               # 이전 소식 확인의 날짜는 미국 동부 날짜로 센다(7절)

# 사용자가 고르는 두 시각(한국 시간, 30분 단위, 미국 서머타임과 상관없이 같은 값). 지금은 변수로 고르고, 나중에는 사용자 입력(파일)에서 읽는다.
#  - 표시 시각: 요약을 보여 줄 시각. 수집 마감은 그 PREP_MIN분 전이다. 수집 노트북(news_collect.ipynb 1절)의 SUMMARY_TIME과 같아야 한다.
#  - 취침 시각: 사용자가 잠드는 시각. 이 시각 뒤에 발행된 기사가 요약에 들어간다. 이 시각 전에 나온 기사(한국 낮·저녁)는
#    요약에 넣지 않고, 밤 기사가 이미 알려진 소식의 후속인지 비교하는 데만 쓴다(7절).
SUMMARY_TIME_OPTIONS = ['06:00', '06:30', '07:00', '07:30', '08:00']
BEDTIME_OPTIONS = ['22:00', '22:30', '23:00', '23:30', '00:00', '00:30', '01:00', '01:30', '02:00']   # 수집 노트북의 BEDTIME_OPTIONS와 같아야 한다(수집은 가장 이른 값부터 1시간마다 검색)
SUMMARY_TIME = '08:00'      # 지금 고른 표시 시각(기본값 08:00)
BEDTIME = '00:00'           # 지금 고른 취침 시각(기본값 00:00)
PREP_MIN = 30               # 수집 마감 = 표시 시각 - 이만큼(분)
assert SUMMARY_TIME in SUMMARY_TIME_OPTIONS and BEDTIME in BEDTIME_OPTIONS, '표시 시각·취침 시각은 고를 수 있는 값 중 하나여야 한다.'
CUTOFF_KST = divmod(int(SUMMARY_TIME[:2]) * 60 + int(SUMMARY_TIME[3:]) - PREP_MIN, 60)   # 수집 마감(시, 분). 08:00 표시면 (7, 30)
CUTOFF_TAG = SUMMARY_TIME.replace(':', '')      # 파일·폴더 이름에 쓰는 표시 시각(예: '0800')
REPORT_BOUNDARY = CUTOFF_KST   # 보고 날짜(폴더) 경계 = 수집 마감. news_collect.ipynb와 같다
# 우리 매체 11곳(news_collect.ipynb 1절과 같다): 출처 주소를 매체 이름으로 맞출 때 쓴다
SITES = ['finance.yahoo.com', 'fool.com', 'cnbc.com', 'theguardian.com', 'businessinsider.com', 'foxbusiness.com',
         'fortune.com', 'cbsnews.com', 'nypost.com', 'usatoday.com', 'cnn.com']
USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0 Safari/537.36'
assert ROOT.name == 'news_search', 'news_search 폴더에서 실행해야 한다.'
print(f'표시 시각 {SUMMARY_TIME}, 수집 마감 {CUTOFF_KST[0]:02d}:{CUTOFF_KST[1]:02d}, 취침 시각 {BEDTIME}')

In [3]:
# ── 묶기 설정 ──
EMBED_MODEL = 'text-embedding-3-large'
EMBED_PRICE = {'text-embedding-3-small': 0.02, 'text-embedding-3-large': 0.13}   # 100만 토큰당 달러(OpenAI 모델 문서, 2026-11)
EMBED_BATCH = 1000          # 요청 1건에 넣을 제목 수(한도 2,048개)
SIM_THRESHOLD = 0.75        # 묶음 안 평균 유사도가 이 이상이면 한 묶음(평균 연결)
CACHE_DIR = ROOT / 'source' / 'embed_cache'      # 제목 → 임베딩 캐시(같은 제목은 다시 요청하지 않는다)

# 대표 기사를 고를 때 검색어 종류의 우선순위(작을수록 직접적). 그래프 연결이 직접적일수록 관련 판단·경로 설명이 쉽다.
KIND_RANK = {'자기 회사': 0,
             '해외 상대': 1, '구체 제품': 1, '브랜드': 1,
             '넓은 제품': 2, '원자재·지표': 2, '정책': 2,
             '테마 대표': 3, '업종 기본': 3, '테마 기본': 3, '회사 기본': 3}
N_TITLES = 3                # 프롬프트에 넘길 제목 수(대표 1 + 후보 2)

mask_key = lambda s: re.sub(r'(sk|org)-[A-Za-z0-9_\-]+', r'\1-***', str(s))

## 2. 작은 도구

보고 날짜 계산(그 기사들로 요약을 만드는 아침의 날짜, 경계는 수집 마감), 보고 날짜의 밤 시각(전날 수집 마감·취침 시각·수집 마감), 보고 날짜 폴더, `.env` 읽기(값은 화면·파일에 쓰지 않음), 텔레그램 알림(실패해도 멈추지 않고 오류에서 토큰을 가림), CSV 이어 쓰기, 쌓인 파일 읽기(`load_night`), 제목·매체 정리. 수집 노트북과 같은 함수를 이 노트북에도 둔다(두 노트북은 쌓인 파일로만 이어진다).

In [ ]:
text_hash = lambda t: hashlib.sha256(t.encode('utf-8')).hexdigest()[:16]
now_utc = lambda: dt.datetime.now(dt.timezone.utc)


def report_id(when=None):
    """보고 날짜: 이 기사들로 요약을 만드는 아침의 날짜. 경계는 수집 마감(REPORT_BOUNDARY, 08:00 표시면 07:30)."""
    t = (when or now_utc()).astimezone(KST)
    if (t.hour, t.minute) >= REPORT_BOUNDARY:
        t += dt.timedelta(days=1)
    return t.strftime('%Y%m%d')


def night_times(nid):
    """보고 날짜 nid의 세 시각(한국 시간): (전날 수집 마감, 취침 시각, 수집 마감).
    전날 수집 마감 ~ 취침 시각에 발행된 기사는 비교용 낮·저녁 묶음, 취침 시각 뒤에 발행되고 수집 마감 전에 받은 기사는 요약에 넣는다.
    취침 시각이 12시 이후(22:00~23:30)면 전날 밤, 그 전(00:00~02:00)이면 보고 날짜 새벽이다."""
    d = dt.datetime.strptime(nid, '%Y%m%d').replace(tzinfo=KST)
    cut = d.replace(hour=CUTOFF_KST[0], minute=CUTOFF_KST[1])
    bh, bm = int(BEDTIME[:2]), int(BEDTIME[3:])
    bed = d.replace(hour=bh, minute=bm) - (dt.timedelta(days=1) if bh >= 12 else dt.timedelta(0))
    return cut - dt.timedelta(days=1), bed, cut


def night_dir(nid):
    d = NIGHTLY_DIR / nid
    d.mkdir(parents=True, exist_ok=True)
    return d


def read_env():
    """프로젝트 루트 .env를 읽는다(값은 화면·파일에 쓰지 않는다)."""
    env = {}
    p = PROJ / '.env'
    if p.exists():
        for line in p.read_text(encoding='utf-8-sig').splitlines():
            k, _, v = line.partition('=')
            env[k.strip()] = v.strip().strip('"\'')
    return env


def notify(text):
    """텔레그램 알림(.env의 ALERT_TELEGRAM_TOKEN·ALERT_TELEGRAM_CHAT_ID). 실패해도 멈추지 않고, 오류에서 토큰을 가린다."""
    env = read_env()
    token, chat = env.get('ALERT_TELEGRAM_TOKEN'), env.get('ALERT_TELEGRAM_CHAT_ID')
    print('알림:', text)
    if token and chat:
        try:
            requests.post(f'https://api.telegram.org/bot{token}/sendMessage', data={'chat_id': chat, 'text': text}, timeout=20)
        except requests.RequestException as e:
            print('  텔레그램 알림 실패:', str(e).replace(token, '***')[:200])


def append_csv(df, path):
    """표를 CSV 끝에 붙인다(처음이면 머리줄과 함께 만든다)."""
    if len(df):
        df.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


def load_night(nid):
    """쌓인 결과 읽기: (기사, 검색어–기사 연결, 요청 기록). 거른 기사는 빼지 않고 drop 칸으로 표시돼 있다."""
    d = NIGHTLY_DIR / nid
    cols = {'items.csv': ['key', 'title', 'source', 'domain', 'outlet', 'published_utc', 'link', 'first_seen_utc', 'drop'],
            'hits.csv': ['key', 'term', 'round', 'group'],
            'fetch_log.csv': ['round', 'label', 'term', 'tier', 'group', 'window_h', 'fetched_utc', 'status', 'n', 'n_in_window', 'capped', 'query']}
    # 파일이 없거나 비어 있어도 칸은 갖춘 빈 표를 돌려준다(뒤 단계가 칸 이름으로 읽는다)
    read = lambda f: pd.read_csv(d / f) if (d / f).exists() else pd.DataFrame(columns=cols[f])
    items = read('items.csv')
    items['drop'] = items['drop'].fillna('')
    return items, read('hits.csv'), read('fetch_log.csv')


def base_outlet(domain):
    """출처 주소 → 우리 매체 이름(지역판 ca.finance.yahoo.com → finance.yahoo.com). 11곳이 아니면 주소 그대로."""
    d = (domain or '').lower()
    return next((s for s in SITES if d == s or d.endswith('.' + s)), d)


def clean_title(title):
    """구글 뉴스 제목 끝의 ' - 매체 이름'을 뗀다."""
    return re.sub(r'\s+-\s+[^-]+$', '', title or '').strip()

## 3. 검색어 표

종목별 대표를 고를 때(검색어 종류)와 그래프 연결 근거를 만들 때(그래프 노드) 쓴다. 칸: `term`(검색 문자열에 넣는 모양), `company`, `company_id`(KR:종목코드), `kind`(검색어 종류), `node`(그래프 노드: 해외 상대 `EN:`, 제품 `P:`, 브랜드 `B:`, 정책 `N:policy:`, 업종 `S:`, 테마 `T:`), `theme`(업종·테마 이름).

- **지금은 임시로** 시험 종목 18개의 검색어(`source/terms/test_queries.csv`)를 쓴다. `graph_db/search_terms` 11절 결과(`graph_db/source/search/test_queries.csv`)의 사본이다. 2026-11-16에는 3차 시험 목록(183개)을 복사해 썼고, 2026-11-17 그래프 표기 통일 뒤 다시 만든 목록(170개)으로 바꿨다. 전체 검색어를 줄이는 규칙을 정하면 실제 검색어 표로 바꾼다.
- **수집 종목**: `COLLECT_COMPANIES`(종목코드 목록)에 있는 종목의 검색어만 쓴다. 지금은 변수로 고르고, 나중에는 사용자 입력(파일)에서 읽는다. 수집·처리 노트북의 목록이 같아야 한다(다르면 처리 쪽이 모르는 검색어로 찾은 기사는 종목에 연결되지 않는다). 수집 노트북과 같은 표여야 한다. 검색어 표를 완성하면 `graph_db/search_terms`의 전체 표에서 보유 종목 것만 읽는다.

In [ ]:
# ── 검색어 표: 시험 종목 검색어 사본(임시, source/terms, news_collect.ipynb 8절과 같은 표) ──
# 수집 종목(종목코드): 이 종목들의 검색어만 검색·처리한다. 지금은 변수로 고르고, 나중에는 사용자 입력(파일)에서 읽는다.
# 두 노트북(news_collect 8절, news_process 3절)이 같아야 한다. 지금 검색어 표(임시)에 있는 종목은 아래 18개뿐이다.
COLLECT_COMPANIES = [
    '000660',  # SK하이닉스
    '003490',  # 대한항공
    '004370',  # 농심
    '005490',  # POSCO홀딩스
    '009540',  # HD한국조선해양
    '012450',  # 한화에어로스페이스
    '042700',  # 한미반도체
    '068270',  # 셀트리온
    '087010',  # 펩트론
    '090430',  # 아모레퍼시픽
    '105560',  # KB금융
    '105840',  # 우진
    '114090',  # GKL
    '257720',  # 실리콘투
    '259960',  # 크래프톤
    '352820',  # 하이브
    '383800',  # LX홀딩스
    '425420',  # 티에프이
]
TQ = pd.read_csv(ROOT / 'source' / 'terms' / 'test_queries.csv', dtype=str).fillna('')
TQ = TQ[TQ['shape'] != '주제만·따옴표 없음']
TERMS = TQ[['query', 'company', 'kind', 'node', 'theme']].rename(columns={'query': 'term'}).drop_duplicates(['term', 'company', 'kind'])
CO = pd.read_csv(PROJ / 'graph_db' / 'source' / 'graph' / 'neo4j' / 'nodes_Company.csv', dtype=str, usecols=['id', 'name']).fillna('')
TERMS['company_id'] = TERMS['company'].map(CO[CO['id'].str.startswith('KR:')].drop_duplicates('name').set_index('name')['id'])   # KR:종목코드
# 수집 종목만 남긴다
want = {'KR:' + c for c in COLLECT_COMPANIES}
no_terms = sorted(c for c in COLLECT_COMPANIES if 'KR:' + c not in set(TERMS['company_id']))
TERMS = TERMS[TERMS['company_id'].isin(want)].copy()
if no_terms:
    print(f'검색어 표에 없는 수집 종목 {len(no_terms)}개(검색하지 않음): {no_terms}')
print(f"검색어 {TERMS['term'].nunique()}개, 종목 {TERMS['company'].nunique()}개 / 종목 ID 없는 줄 {TERMS['company_id'].isna().sum()}")

## 4. 묶기

- **임베딩**: 제목을 `text-embedding-3-large`로 임베딩한다. 같은 제목은 캐시에서 꺼내고(다시 요청하지 않음), 캐시에 없는 제목만 요청한다. `EMBED_RUN = False`면 요청하지 않고 예상 비용을 알리며 멈춘다. 오류 메시지에 키가 섞여도 가린다.
- **묶기**: 평균 연결 0.75(두 묶음 사이 모든 쌍의 평균 유사도가 기준 이상이면 합침). 시험에서 사건 단위로 나뉘었다(황 사임 발표 / 해임 검토 / 임시 CEO / 주가 / 후임 후보가 따로).
- **중요도**: 서로 다른 매체 수, 같으면 기사 수(기사 수만 보면 한 매체의 연재형 기사·야후 재게재가 부풀린다). 야후가 다른 매체 기사를 다시 실은 것(같은 묶음 안에서 제목이 같음)은 원래 매체로 세고 기사 수에서도 뺀다(`outlet_counted`, 2026-11-17).
- **종목별 대표 기사**: 그 종목의 검색어로 걸린 기사 중 ① 제목에 검색어가 있는 것 ② 더 직접적인 검색어(자기 회사 > 해외 상대·구체 제품·브랜드 > 넓은 제품·원자재·정책 > 테마 대표·업종·테마·회사 기본) ③ 묶음 중심에 가까운 것 ④ 먼저 나온 것. 같은 묶음이라도 종목마다 그 종목이 실제로 찾은 기사로 판단하게 해, 제목 틀이 같은 연재 기사가 한 묶음이 되어도 관련 기사가 묻히지 않게 한다.
- **후보 제목**: 그 종목 후보의 다음 순위 2개, 후보가 하나뿐이면 묶음 중심에 가까운 제목 2개.
- **묶음별 본문 기사**: 가장 많은 종목이 대표로 고른 기사(같으면 중심에 가까운 것). 프롬프트 요청은 묶음마다 이 본문 1건을 기본으로 하고, 대표가 다른 소식인 종목만 따로 보낸다(`nightly_prompt_requirements.md` 2-2).

In [6]:
def _cache_path(model):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return CACHE_DIR / f'{model}.npz'


def load_cache(model=EMBED_MODEL):
    """캐시 → {제목 해시: 벡터}."""
    p = _cache_path(model)
    if not p.exists():
        return {}
    z = np.load(p)
    return dict(zip(z['keys'].tolist(), z['vecs']))


def save_cache(cache, model=EMBED_MODEL):
    keys = list(cache)
    np.savez(_cache_path(model), keys=np.array(keys), vecs=np.stack([cache[k] for k in keys]).astype('float32'))


def embed(texts, model=EMBED_MODEL, run=True, log=print):
    """제목 목록 → 정규화한 임베딩 행렬(행 = 제목). 캐시에 없는 제목만 OpenAI에 요청한다.

    run=False면 요청하지 않고, 캐시에 없는 제목이 있으면 알리고 멈춘다(비용이 드는 요청은 스위치를 켜야 보낸다).
    요청할 때마다 토큰과 비용을 embed_cache/runs.csv에 남긴다.
    """
    cache = load_cache(model)
    keys = [text_hash(t) for t in texts]
    todo = sorted({k: t for k, t in zip(keys, texts) if k not in cache}.items())
    if todo:
        if not run:
            raise RuntimeError(f'캐시에 없는 제목 {len(todo)}개: 임베딩 요청이 필요하다(run=True, 예상 비용 약 ${len(todo) * 20 * EMBED_PRICE[model] / 1e6:.4f})')
        from openai import OpenAI
        client = OpenAI(api_key=read_env().get('OPENAI_API_KEY'))
        tokens = 0
        try:
            for i in range(0, len(todo), EMBED_BATCH):
                part = todo[i:i + EMBED_BATCH]
                r = client.embeddings.create(model=model, input=[t for _, t in part])
                for (k, _), d in zip(part, r.data):
                    cache[k] = np.array(d.embedding, dtype='float32')
                tokens += r.usage.total_tokens
        except Exception as e:                       # 오류 메시지에 키가 섞여 나와도 가린다
            raise RuntimeError('임베딩 요청 실패: ' + mask_key(e)[:300]) from None
        finally:
            save_cache(cache, model)
        cost = tokens * EMBED_PRICE[model] / 1e6
        append_csv(pd.DataFrame([dict(utc=dt.datetime.now(dt.timezone.utc).isoformat(), model=model, n=len(todo),
                                      requests=-(-len(todo) // EMBED_BATCH), tokens=tokens, cost_usd=round(cost, 5))]), CACHE_DIR / 'runs.csv')
        log(f'임베딩 요청: 제목 {len(todo)}개, 토큰 {tokens:,}, 비용 ${cost:.4f}')
    X = np.stack([cache[k] for k in keys])
    return X / np.linalg.norm(X, axis=1, keepdims=True)

In [ ]:
def cluster_labels(X, threshold=SIM_THRESHOLD):
    """임베딩 → 묶음 번호. 평균 연결: 두 묶음 사이 모든 쌍의 평균 유사도가 기준 이상이면 합친다."""
    if len(X) < 2:
        return np.zeros(len(X), dtype=int)
    from sklearn.cluster import AgglomerativeClustering
    return AgglomerativeClustering(n_clusters=None, distance_threshold=1 - threshold,
                                   metric='cosine', linkage='average').fit_predict(X)


def build_clusters(items, run=True, log=print):
    """기사(거르기를 통과한 것) → (기사에 묶음 번호·중심 유사도를 붙인 표, 묶음 표).

    묶음 표: 묶음 번호, 기사 수, 서로 다른 매체 수, 매체 목록, 가장 이른·늦은 발행 시각, 중심에 가장 가까운 제목, 중요도 순위.
    중요도 순위는 서로 다른 매체 수 → 기사 수 → 이른 발행 순이다.
    """
    items = items.reset_index(drop=True).copy()
    X = embed(items['title'].tolist(), run=run, log=log)
    items['cluster'] = cluster_labels(X)
    # 묶음 중심(평균 벡터)과 각 기사의 유사도: 묶음을 가장 잘 대표하는 기사를 고를 때 쓴다
    centers = {c: X[idx].mean(0) for c, idx in items.groupby('cluster').indices.items()}
    items['center_sim'] = [float(X[i] @ centers[c] / np.linalg.norm(centers[c])) for i, c in enumerate(items['cluster'])]
    return items, cluster_table(items)


YAHOO = 'finance.yahoo.com'
title_norm = lambda t: re.sub(r'[^a-z0-9]+', ' ', str(t).lower().replace('\u2019', "'").replace('\u2018', "'")).strip()


def outlet_counted(items):
    """매체 수·기사 수를 셀 때 쓰는 매체와 다시 싣기 여부. 야후가 다른 매체 기사를 다시 실은 것(같은 묶음 안에서 다른 매체 기사와
    제목이 같음, 대소문자·따옴표 모양·문장부호는 무시)은 원래 매체로 세고 기사 수에서 뺀다. 11-16 1차에서 매체 2곳 이상 묶음 44개 중
    42개가 이런 다시 싣기(41개가 Motley Fool)라 중요도가 부풀려졌다. 반환: (센 매체, 다시 싣기 여부) 두 열."""
    norm = items['title'].map(title_norm)
    orig = {}
    for c, n, o in zip(items['cluster'], norm, items['outlet']):
        if o != YAHOO:
            orig.setdefault((c, n), o)
    counted = [orig.get((c, n), o) if o == YAHOO else o for c, n, o in zip(items['cluster'], norm, items['outlet'])]
    repost = [o == YAHOO and (c, n) in orig for c, n, o in zip(items['cluster'], norm, items['outlet'])]
    return pd.Series(counted, index=items.index), pd.Series(repost, index=items.index)


def cluster_table(items):
    """기사 표(cluster·center_sim 칸) → 묶음 표: 기사 수, 서로 다른 매체 수, 매체 목록, 가장 이른·늦은 발행 시각, 중심에 가장 가까운 제목, 중요도 순위.
    중요도 순위는 서로 다른 매체 수 → 기사 수 → 이른 발행 순이다. 야후가 다시 실은 기사는 원래 매체로 세고 기사 수에서 뺀다(n_reposts에 따로 셈)."""
    items = items.copy()
    items['outlet_counted'], items['repost'] = outlet_counted(items)   # 야후 다시 싣기는 원래 매체로, 기사 수에서 뺌
    g = items.groupby('cluster')
    C = pd.DataFrame({
        'n_articles': g['repost'].apply(lambda s: int((~s).sum())),
        'n_outlets': g['outlet_counted'].nunique(),
        'outlets': g['outlet_counted'].agg(lambda s: ', '.join(sorted(set(s)))),
        'n_reposts': g['repost'].sum().astype(int),
        'first_published_utc': g['published_utc'].min(),
        'last_published_utc': g['published_utc'].max(),
        'title': g.apply(lambda d: d.loc[d['center_sim'].idxmax(), 'title'], include_groups=False),
    }).reset_index()
    C = C.sort_values(['n_outlets', 'n_articles', 'first_published_utc'], ascending=[False, False, True]).reset_index(drop=True)
    C['rank'] = range(1, len(C) + 1)
    return C

In [8]:
def term_in_title(term, title):
    """검색어가 제목에 있나(대소문자 무시). 본문에만 스친 기사보다 그 기사의 주제일 가능성이 크다.

    따옴표 구문이 여러 개면("Nvidia" "memory chips") 모두 있어야 한다.
    """
    phrases = re.findall(r'"([^"]+)"', term) or [term]
    t = (title or '').lower()
    return all(p.lower() in t for p in phrases)


def pick_representatives(items, hits, terms, n_titles=N_TITLES):
    """종목마다, 그 종목이 걸린 묶음마다 대표 기사 1건과 후보 제목을 고른다.

    items: build_clusters가 묶음 번호를 붙인 기사 표(key, title, outlet, published_utc, cluster, center_sim)
    hits:  검색어–기사 연결(key, term)
    terms: 검색어 표(term, company, kind)
    반환 칸: company, cluster, rep_key, rep_title, rep_outlet, rep_terms(이 기사를 찾은 그 종목의 검색어), rep_kind,
            other_titles(후보 제목, ' || '로 구분), n_candidates(그 종목의 후보 기사 수)
    """
    links = (hits[['key', 'term']].drop_duplicates()
             .merge(terms[['term', 'company', 'kind']].drop_duplicates(), on='term')
             .merge(items[['key', 'title', 'outlet', 'published_utc', 'cluster', 'center_sim']], on='key'))
    links['in_title'] = [term_in_title(t, ti) for t, ti in zip(links['term'], links['title'])]
    links['kind_rank'] = links['kind'].map(KIND_RANK).fillna(9)
    # 기사 하나가 한 종목의 여러 검색어에 걸렸으면 가장 유리한 것으로 본다(제목에 있음, 가장 직접적인 종류)
    cand = (links.groupby(['company', 'cluster', 'key'])
            .agg(title=('title', 'first'), outlet=('outlet', 'first'), published_utc=('published_utc', 'first'),
                 center_sim=('center_sim', 'first'), in_title=('in_title', 'max'), kind_rank=('kind_rank', 'min'),
                 terms=('term', lambda s: ', '.join(sorted(set(s)))), kinds=('kind', lambda s: ', '.join(sorted(set(s)))))
            .reset_index())
    # 고르는 순서: ① 제목에 검색어 ② 직접적인 검색어 ③ 묶음 중심에 가까움 ④ 먼저 나옴
    cand = cand.sort_values(['company', 'cluster', 'in_title', 'kind_rank', 'center_sim', 'published_utc'],
                            ascending=[True, True, False, True, False, True])
    central = items.sort_values('center_sim', ascending=False).groupby('cluster')['title'].apply(list).to_dict()
    rows = []
    for (company, cluster), g in cand.groupby(['company', 'cluster'], sort=False):
        rep = g.iloc[0]
        others = g['title'].iloc[1:n_titles].tolist()
        if not others:                 # 후보가 대표 하나뿐이면 묶음 중심에 가까운 제목으로 채운다
            others = [t for t in central[cluster] if t != rep['title']][:n_titles - 1]
        rows.append(dict(company=company, cluster=cluster, rep_key=rep['key'], rep_title=rep['title'], rep_outlet=rep['outlet'],
                         rep_terms=rep['terms'], rep_kind=rep['kinds'], rep_in_title=bool(rep['in_title']),
                         other_titles=' || '.join(others), n_candidates=len(g)))
    return pd.DataFrame(rows)


def cluster_body_article(S, items):
    """묶음마다 본문을 받을 기사 1건(요청 단위 방법 A): 가장 많은 종목이 대표로 고른 기사, 같으면 묶음 중심에 가까운 기사.

    종목별 대표가 이 기사와 다른 소식이면 밤사이 프롬프트가 표시하고, 그 종목만 자기 대표 본문으로 다시 요청한다
    (nightly_prompt_requirements.md 2-2).
    """
    sim = items.set_index('key')['center_sim']
    votes = S.groupby(['cluster', 'rep_key']).size().rename('n_companies').reset_index()
    votes['center_sim'] = votes['rep_key'].map(sim)
    best = votes.sort_values(['cluster', 'n_companies', 'center_sim'], ascending=[True, False, False]).drop_duplicates('cluster')
    return best.rename(columns={'rep_key': 'body_key'})[['cluster', 'body_key', 'n_companies']]

## 5. 본문 받기

밤사이 프롬프트에 넣을 본문을 받는다. 묶음마다 본문 기사 1건만 받는다(기사 전부를 받으면 요청이 수천 건이라).

1. **원문 주소 복원**: 구글 뉴스 링크 → 원문 주소(`googlenewsdecoder`). 40건씩 4개 동시, 간격 0.2초. 언론사 시험 때 이 설정으로 6,453건을 막힘 없이 복원했다(검색과 한도가 따로).
2. **원문 페이지 받기**: 매체마다 0.5초 간격으로 차례로, 매체끼리는 동시에 받는다.
3. **본문 추출**: `trafilatura`.
4. **거르기 2·3단계**(`judge_page`, 언론사 시험의 `classify_page`와 같은 규칙): 매체별 제외 구역(팬 사이트, 보도자료 구역 등), 야후 시세 페이지(`/quote/`), 영상·라이브, 보도자료, 기사 아님(JSON-LD 기사 유형·작성 시각이 없음), 유료벽, 너무 짧음(80단어 미만), 일부만 받음. `SUCCESS`인 것만 본문으로 쓴다. 언론사 시험 때 받은 HTML 600건으로 대조해 99% 같은 판정(다른 5건은 야후 시세 페이지를 이번에 뺀 것).

원문 주소·HTML은 `source/bodies/`에 캐시한다(같은 기사는 다시 받지 않음).

In [9]:
# ── 본문 받기: 원문 주소 복원 → 원문 페이지 받기 → 본문 추출 → 거르기 2·3단계 ──
import asyncio
import concurrent.futures as cf
import threading
from urllib.parse import urlparse

from dateutil import parser as date_parser
from googlenewsdecoder import gnews_decoder_async
from lxml import html as lxml_html
from trafilatura import extract
from trafilatura.utils import decode_file

BODY_DIR = ROOT / 'source' / 'bodies'           # 기사마다 한 번만 받는다(보고 날짜와 상관없이 같은 캐시)
RESOLVE_CACHE = BODY_DIR / 'resolve_cache.jsonl'  # 구글 링크 → 원문 주소
FETCH_CACHE = BODY_DIR / 'fetch_cache.jsonl'      # 원문 페이지 받은 기록
HTML_DIR = BODY_DIR / 'html'                       # 원문 HTML(gzip)
RESOLVE_BATCH, RESOLVE_CONCURRENCY, RESOLVE_INTERVAL = 40, 4, 0.2   # 언론사 시험 때 6,453건을 막힘 없이 복원한 설정
FETCH_SLEEP = 0.5            # 같은 매체에 보내는 요청 사이 간격(초). 매체끼리는 동시에 받는다
FETCH_TIMEOUT = 15
MIN_BODY_WORDS = 80          # 본문이 이보다 짧으면 기사로 보지 않는다(단신은 통과하도록 낮게)
MIN_LD_COVERAGE = 0.6        # JSON-LD 본문 대비 추출 본문 비율이 이보다 낮으면 일부만 받은 것
MAX_BODY_TRIES = 3           # 묶음 하나에서 본문 기사를 받아 보는 최대 수(앞 기사가 실패하면 다음 기사로)

ARTICLE_LD_TYPES = {'Article', 'NewsArticle', 'ReportageNewsArticle', 'AnalysisNewsArticle', 'OpinionNewsArticle',
                    'BackgroundNewsArticle', 'ReviewNewsArticle', 'BlogPosting', 'Report'}
PAYWALL_PHRASES = ['subscribe to continue', 'subscribe to read', 'subscription required', 'to continue reading',
                   'already a subscriber', 'sign in to continue', 'sign in to read', 'log in to continue',
                   'create a free account to continue', 'this article is for subscribers', 'this content is for subscribers',
                   'unlock this article', 'become a member to read']
BLOCK_TITLE_PHRASES = ['access denied', 'just a moment', 'attention required', 'are you a robot', 'pardon our interruption',
                       'request unsuccessful', 'verify you are human', '403 forbidden']
BLOCK_HTML_MARKERS = ['px-captcha', 'captcha-delivery.com', 'cf-chl-', '_incapsula_resource']
PRESS_RELEASE_PROVIDERS = ['globenewswire', 'globe newswire', 'pr newswire', 'prnewswire', 'business wire', 'accesswire',
                           'access newswire', 'newsfile', 'eqs news']
PRESS_RELEASE_BODY = re.compile(r'PR ?Newswire|Business ?Wire|Globe ?Newswire|ACCESS ?Newswire|ACCESSWIRE|Newsfile|EQS-News|/CNW/', re.I)
# 거르기 2단계(원문 주소): 영상·라이브·시세 페이지, 매체별 제외 구역(언론사 시험과 같다)
VIDEO_PATH = re.compile(r'/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)', re.I)
VIDEO_HOST = re.compile(r'^(video|videos|watch|tv)\.', re.I)
LIVE_PATH = re.compile(r'/(live|live-news|live-updates|liveblog)(/|$)|live-updates', re.I)
QUOTE_PATH = re.compile(r'finance\.yahoo\.com/quote/|/price-of-|/industry-statistics/', re.I)
SECTION_EXCLUDE_RULES = {
    'businessinsider.com': [r'^markets\.', r'^africa\.'], 'usatoday.com': [r'^ftw\.', r'^patriotswire\.', r'/story/shopping/', r'/press-release/'],
    'cnbc.com': [r'/investingclub/'], 'cnn.com': [r'^transcripts\.', r'/cnn-underscored/', r'^cnnicevents\.'], 'nypost.com': [r'/betting/'],
}

host_of = lambda u: (urlparse(u or '').netloc or '').lower().removeprefix('www.')
url_hash = lambda u: hashlib.sha256((u or '').encode('utf-8')).hexdigest()[:24]


def _read_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()] if path.exists() else []


def _append_jsonl(path, rows):
    if rows:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('a', encoding='utf-8') as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + '\n')


def resolve_urls(google_urls, log=print):
    """구글 뉴스 링크 → 원문 주소. 캐시에 없는 것만 googlenewsdecoder로 복원한다(검색과 한도가 따로).
    노트북(이벤트 루프가 이미 도는 곳)에서도 쓸 수 있게 별도 스레드에서 asyncio를 돌린다."""
    cache = {r['google_url']: r for r in _read_jsonl(RESOLVE_CACHE) if r.get('original_url')}
    todo = [u for u in dict.fromkeys(google_urls) if u not in cache]
    for i in range(0, len(todo), RESOLVE_BATCH):
        batch = todo[i:i + RESOLVE_BATCH]
        coro = lambda: gnews_decoder_async(batch, interval=RESOLVE_INTERVAL, timeout=15.0, concurrency=RESOLVE_CONCURRENCY)
        try:
            with cf.ThreadPoolExecutor(1) as ex:
                results = ex.submit(asyncio.run, coro()).result()
            results = [results] if isinstance(results, dict) else results
        except Exception as e:
            results = [{'success': False, 'message': f'{type(e).__name__}: {e}'}] * len(batch)
        rows = [dict(google_url=u, original_url=r.get('decoded_url') if r.get('success') else None,
                     error=None if r.get('success') else str(r.get('message'))[:200], resolved_utc=now_utc().isoformat())
                for u, r in zip(batch, results)]
        _append_jsonl(RESOLVE_CACHE, rows)
        cache.update({r['google_url']: r for r in rows})
    if todo:
        log(f'  원문 주소 복원 {len(todo)}건(성공 {sum(1 for u in todo if cache[u].get("original_url"))})')
    return {u: cache[u].get('original_url') for u in google_urls if u in cache}


def fetch_pages(urls, log=print):
    """원문 페이지 받기(캐시). 매체마다 FETCH_SLEEP 간격으로 차례로, 매체끼리는 동시에 받는다."""
    cache = {r['url']: r for r in _read_jsonl(FETCH_CACHE) if r.get('status') == 'OK'}
    todo = [u for u in dict.fromkeys(urls) if u and u not in cache]
    by_host = {}
    for u in todo:
        by_host.setdefault(host_of(u), []).append(u)
    HTML_DIR.mkdir(parents=True, exist_ok=True)
    lock = threading.Lock()
    out_rows = []

    def worker(host_urls):
        ses = requests.Session()
        ses.headers.update({'User-Agent': USER_AGENT,
                            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-US,en;q=0.9'})
        for u in host_urls:
            rec = dict(url=u, final_url=None, http=None, status=None, fetched_utc=now_utc().isoformat())
            try:
                r = ses.get(u, timeout=FETCH_TIMEOUT, allow_redirects=True)
                ctype = r.headers.get('Content-Type', '').lower()
                rec.update(final_url=r.url, http=r.status_code,
                           status='OK' if r.status_code == 200 and 'html' in ctype else ('NOT_HTML' if r.status_code == 200 else 'HTTP_ERROR'))
                if 'html' in ctype:
                    (HTML_DIR / f'{url_hash(u)}.html.gz').write_bytes(gzip.compress(r.content))
            except requests.RequestException as e:
                rec.update(status='TIMEOUT' if isinstance(e, requests.Timeout) else 'REQUEST_ERROR')
            with lock:
                out_rows.append(rec)
            time.sleep(FETCH_SLEEP)

    with cf.ThreadPoolExecutor(max_workers=max(1, min(12, len(by_host)))) as ex:
        list(ex.map(worker, by_host.values()))
    _append_jsonl(FETCH_CACHE, out_rows)
    cache.update({r['url']: r for r in out_rows})
    if todo:
        log(f'  원문 페이지 {len(todo)}건(매체 {len(by_host)}곳, 성공 {sum(r["status"] == "OK" for r in out_rows)})')
    return {u: cache.get(u) for u in urls if u}


def read_html(url):
    p = HTML_DIR / f'{url_hash(url)}.html.gz'
    return decode_file(gzip.decompress(p.read_bytes())) if p.exists() else None


def _ld_items(tree):
    items = []
    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            stack = [json.loads(node.text_content().strip())]
        except Exception:
            continue
        while stack:
            cur = stack.pop()
            if isinstance(cur, list):
                stack.extend(cur)
            elif isinstance(cur, dict):
                items.append(cur)
                if '@graph' in cur:
                    stack.append(cur['@graph'])
    return items


def _types(item):
    t = item.get('@type', [])
    return {x for x in ([t] if isinstance(t, str) else t) if isinstance(x, str)}


def page_features(raw_html, final_url):
    """원문 HTML → 판정에 쓰는 특징과 본문(언론사 시험의 extract_page_features를 줄인 것)."""
    tree = lxml_html.fromstring(re.sub(r'^\s*<\?xml[^>]*\?>', '', raw_html))
    title = (tree.findtext('.//title') or '').strip()
    og = tree.xpath('//meta[@property="og:type"]/@content')
    og_type = og[0].strip().lower() if og else None
    items = _ld_items(tree)
    ld_types = set().union(*[_types(i) for i in items]) if items else set()
    arts = [i for i in items if _types(i) & ARTICLE_LD_TYPES]
    ld_words = max((len(str(i.get('articleBody') or '').split()) for i in arts), default=0)
    provider = ''
    for i in arts:
        for v in (i.get('provider'), i.get('sourceOrganization'), i.get('author')):
            v = v[0] if isinstance(v, list) and v else v
            if isinstance(v, dict) and v.get('@type') in (None, 'Organization', 'NewsMediaOrganization') and v.get('name'):
                provider = provider or v['name']
    published = next((i.get('datePublished') for i in arts if i.get('datePublished')), None) \
        or next(iter(tree.xpath('//meta[@property="article:published_time"]/@content')), None)
    body = (extract(raw_html, url=final_url, include_comments=False, include_tables=False, favor_precision=True) or '').strip()
    low = body.lower()
    return dict(page_title=title, og_type=og_type, ld_has_article=bool(arts), ld_has_video='VideoObject' in ld_types,
                ld_has_liveblog='LiveBlogPosting' in ld_types, ld_words=ld_words, provider=provider, published=published,
                blocked=any(p in title.lower() for p in BLOCK_TITLE_PHRASES) or any(m in raw_html[:50000].lower() for m in BLOCK_HTML_MARKERS),
                paywall=next((p for p in PAYWALL_PHRASES if p in low), None),
                press_release=any(n in provider.lower() for n in PRESS_RELEASE_PROVIDERS) or bool(PRESS_RELEASE_BODY.search(body[:600])),
                body=body, body_words=len(body.split()))


def judge_page(outlet, url, fetch, f):
    """원문 페이지 판정: 'SUCCESS'면 본문을 쓴다. 아니면 사유(거르기 2·3단계, 받기 실패)."""
    target = host_of(url) + urlparse(url or '').path
    if any(re.search(r, target) for r in SECTION_EXCLUDE_RULES.get(outlet, [])):
        return 'EXCLUDED_SECTION'
    if QUOTE_PATH.search(url or ''):
        return 'QUOTE_PAGE'
    if fetch is None or fetch.get('status') != 'OK':
        return (fetch or {}).get('status') or 'NOT_FETCHED'
    final = fetch.get('final_url') or url
    if not (host_of(final) == outlet or host_of(final).endswith('.' + outlet)):
        return 'REDIRECTED_AWAY'
    if f is None:
        return 'NOT_FETCHED'
    if f['blocked']:
        return 'BLOCKED'
    path = urlparse(final).path
    if VIDEO_PATH.search(path) or VIDEO_HOST.search(host_of(final)) or (f['og_type'] or '').startswith('video') \
            or (f['ld_has_video'] and not f['ld_has_article']):
        return 'VIDEO_PAGE'
    if LIVE_PATH.search(path) or f['ld_has_liveblog']:
        return 'LIVE_BLOG'
    if f['press_release']:
        return 'PRESS_RELEASE'
    if not f['ld_has_article'] or f['og_type'] == 'website' or not f['published']:
        return 'NOT_ARTICLE'
    if not f['body']:
        return 'EXTRACT_FAILED'
    if f['paywall']:
        return 'PAYWALL'
    if f['body_words'] < MIN_BODY_WORDS:
        return 'TOO_SHORT'
    if f['ld_words'] and f['body_words'] / f['ld_words'] < MIN_LD_COVERAGE:
        return 'PARTIAL_BODY'
    return 'SUCCESS'


def get_bodies(items, log=print):
    """기사(key, link, outlet) → 본문 표: key, original_url, final_url, status, body, body_words, page_title, provider."""
    t0 = time.time()
    orig = resolve_urls(items['link'].tolist(), log=log)
    t1 = time.time()
    fetch = fetch_pages([orig.get(u) for u in items['link']], log=log)
    t2 = time.time()
    rows = []
    for r in items.itertuples():
        url = orig.get(r.link)
        fr = fetch.get(url) if url else None
        raw = read_html(url) if fr and fr.get('status') == 'OK' else None
        try:
            f = page_features(raw, fr['final_url']) if raw else None
        except Exception:
            f = None
        status = 'NOT_RESOLVED' if not url else judge_page(r.outlet, url, fr, f)
        rows.append(dict(key=r.key, original_url=url, final_url=(fr or {}).get('final_url'), status=status,
                         body=f['body'] if f and status == 'SUCCESS' else '', body_words=f['body_words'] if f else 0,
                         page_title=f['page_title'] if f else '', provider=f['provider'] if f else ''))
    B = pd.DataFrame(rows)
    B.attrs['seconds'] = dict(resolve=round(t1 - t0, 1), fetch=round(t2 - t1, 1), extract=round(time.time() - t2, 1))
    return B

## 6. 그래프 연결 근거

밤사이 프롬프트가 "이 기사가 이 종목과 관련 있나"를 판단할 근거다. 종목이 그 검색어로 기사를 찾은 이유를 그래프에서 문장으로 만든다(`graph_evidence`).

| 검색어 종류 | 근거 |
|---|---|
| 자기 회사 | 종목 자신에 관한 기사 |
| 해외 상대·브랜드·원자재 등 | 종목과 그 노드 사이의 관계(양쪽 방향): "SK하이닉스 → 공급 → Nvidia (HBM) — 메모 [DART 공시]" |
| 구체·넓은 제품 | 직접 만드는 관계, 없으면 종목이 만드는 제품의 위 계층(2단계까지): "… → 생산 → HBM3E (위 계층: high-bandwidth memory)" |
| 정책 | 정책이 종목의 제품·원자재에 걸리는 관계 |
| 테마 대표, 업종·테마·회사 기본 | 종목의 테마·업종에 관한 산업 기사(종목 직접 연결 아님) |

관계 문장에는 관계 종류(우리말), 관계 제품, 그래프 만들 때 남긴 메모(근거 요약), 출처(DART 공시/미국 뉴스)를 넣는다. 종목–검색어 하나에 최대 3개.

In [10]:
# ── 그래프 연결 근거: 종목이 그 검색어로 기사를 찾은 이유를 그래프에서 문장으로 ──
GRAPH_DIR = PROJ / 'graph_db' / 'source' / 'graph' / 'neo4j'     # graph_db가 만든 노드·관계 표
REL_NAMES = {'PRODUCES': '생산', 'SUPPLIES': '공급', 'BUYS': '구매·사용', 'COMPETES': '경쟁', 'PARTNERS': '협력',
             'INVESTS': '투자', 'OWNS': '지분', 'LICENSES': '기술 이전', 'WINS_ORDER': '수주', 'DISPUTES': '분쟁',
             'BUILDS_OPERATES': '시설 건설·운영', 'AFFECTS': '영향', 'REGULATES': '규제', 'FUNDS': '자금 지원',
             'DIVESTS': '매각', 'OTHER': '기타 계약·관계', 'HAS_BRAND': '브랜드', 'POLICY_STANCE': '정책 입장'}
SOURCE_NAMES = {'dart': 'DART 공시', 'news': '미국 뉴스'}
MAX_EVIDENCE = 3            # 종목–검색어 하나에 넘길 근거 문장 수
_GRAPH = {}


def load_graph():
    """관계 표(종류별 rels_*.csv)와 노드 이름을 한 번만 읽어 둔다."""
    if _GRAPH:
        return _GRAPH
    loads = lambda s: json.loads(s) if isinstance(s, str) and s.startswith('[') else []
    rels = []
    for f in GRAPH_DIR.glob('rels_*.csv'):
        t = f.stem[5:]
        if t not in REL_NAMES and t != 'SUBTYPE_OF':
            continue
        r = pd.read_csv(f, dtype=str, usecols=lambda c: c in ('start', 'end', 'sources', 'notes', 'products')).fillna('')
        r['type'] = t
        rels.append(r)
    R = pd.concat(rels, ignore_index=True)
    for c in ('sources', 'notes', 'products'):
        R[c] = R[c].map(loads) if c in R else [[]] * len(R)
    names = {}                                       # 노드 ID → 보여 줄 이름: 국내 기업은 한글 이름, 나머지는 영어 이름(없으면 이름)
    for f in GRAPH_DIR.glob('nodes_*.csv'):
        n = pd.read_csv(f, dtype=str, usecols=lambda c: c in ('id', 'name', 'name_en')).fillna('')
        en = n['name_en'] if 'name_en' in n else pd.Series('', index=n.index)
        ko = n['name'] if 'name' in n else pd.Series('', index=n.index)
        shown = en.where(en != '', ko)
        shown = ko.where(n['id'].str.startswith('KR:') & (ko != ''), shown)
        names.update(dict(zip(n['id'], shown)))
    _GRAPH.update(R=R, names=names, parent=R[R['type'] == 'SUBTYPE_OF'].groupby('start')['end'].agg(list).to_dict())
    return _GRAPH


def _sentence(row, company_id):
    """관계 한 줄 → 근거 문장: "SK하이닉스 → 공급 → Nvidia (HBM) — 메모: … [DART 공시]"."""
    nm = load_graph()['names']
    a, b = nm.get(row.start, row.start), nm.get(row.end, row.end)
    prods = ', '.join(nm.get(p, p.split(':')[-1]) for p in row.products[:2])
    note = ' / '.join(row.notes[:2])
    src = ', '.join(SOURCE_NAMES.get(s, s) for s in row.sources)
    return f"{a} → {REL_NAMES[row.type]} → {b}" + (f" ({prods})" if prods else '') + (f" — {note}" if note else '') + (f" [{src}]" if src else '')


def ancestors(pid, depth=2):
    par, out, cur = load_graph()['parent'], [], [pid]
    for _ in range(depth):
        cur = [q for p in cur for q in par.get(p, [])]
        out += cur
    return out


def graph_evidence(company_id, node, kind, theme=''):
    """종목(KR:종목코드)이 검색어 노드(node)로 이어지는 근거 문장 목록(최대 MAX_EVIDENCE개).

    - 자기 회사·업종·테마·회사 기본: 그래프 관계가 아니라 검색어 종류가 근거다.
    - 해외 상대·브랜드·원자재·정책 등: 종목과 노드 사이의 관계(양쪽 방향)를 문장으로.
    - 제품: 직접 만드는 관계가 없으면, 종목이 만드는 제품의 위 계층(2단계까지)에 그 제품이 있는지.
    - 정책: 정책이 종목의 제품·원자재에 걸리는 관계.
    """
    nm = load_graph()['names']
    me = nm.get(company_id, company_id)
    if kind == '자기 회사':
        return [f'{me} 자신에 관한 기사(회사 이름으로 찾음)']
    if kind in ('업종 기본', '테마 기본', '회사 기본'):
        what = {'업종 기본': f"{me}의 업종 '{theme}'", '테마 기본': f"{me}의 테마 '{theme}'", '회사 기본': f'{me}의 주요 사업'}[kind]
        return [f'{what}에 관한 산업 기사(기본 검색어로 찾음, 종목 직접 연결 아님)']
    R = load_graph()['R']
    direct = R[((R['start'] == company_id) & (R['end'] == node)) | ((R['start'] == node) & (R['end'] == company_id))]
    out = [_sentence(r, company_id) for r in direct.itertuples() if r.type in REL_NAMES]
    if not out and node.startswith('P:'):
        mine = R[(R['start'] == company_id) & (R['type'] == 'PRODUCES')]
        for r in mine.itertuples():
            if node in ancestors(r.end):
                out.append(f"{me} → 생산 → {nm.get(r.end, r.end)} (위 계층: {nm.get(node, node)})")
    if not out and node.startswith('N:policy:'):
        mine = set(R.loc[(R['start'] == company_id) & R['type'].isin(['PRODUCES', 'BUYS']), 'end'])
        out = [_sentence(r, company_id) for r in R[(R['start'] == node) & R['end'].isin(mine)].itertuples() if r.type in REL_NAMES]
    if not out and kind == '테마 대표':
        out = [f"{me}의 테마 '{theme}' 종목들이 만드는 대표 제품 {nm.get(node, node)}(종목 직접 연결 아님)"]
    return out[:MAX_EVIDENCE] or [f'{nm.get(node, node)}: 그래프에 직접 관계가 없음(검색어 종류 {kind})']

## 7. 이전 소식 확인

요약에 넣을 묶음(제목이 비슷한 기사끼리 모은 것)마다, **이미 알려진 소식의 후속인지, 처음 나온 소식인지**를 정한다. 제목만 보는 첫 판단이고, 정말 새 진전인지 같은 말의 반복인지는 밤사이 프롬프트가 지난 묶음 제목과 비교해 최종으로 정한다.

- **비교 방법**: 이번 묶음의 중심 벡터(묶음 안 기사 제목 벡터의 평균)를 **지난 묶음 기록**의 중심 벡터들과 비교한다. 가장 비슷한 것이 0.75 이상이면 `이전 소식의 후속`(그 묶음의 사건 번호를 그대로 쓰고, 지난 묶음 제목을 함께 남긴다), 아니면 `처음 나온 소식`(새 사건 번호).
- **사건 번호**: 같은 사건을 다룬 묶음들에 붙이는 번호다. 날짜가 달라도 같은 사건이면 같은 번호가 이어진다.
- **비교 기간**: 이번 묶음에서 가장 먼저 나온 기사의 **미국 동부 날짜**에서 2일 전 00:00부터. 예: 16일 기사로 된 묶음은 14일 00:00 뒤에 발행된 기사가 있는 지난 묶음과 비교한다(14·15일 전체와 16일 앞서 나온 것). 지난 묶음은 그 묶음의 가장 늦은 발행 시각으로 기간 안인지 본다.
- **비교 대상인 지난 묶음**은 두 가지다.
  1. **비교용 낮·저녁 묶음**: 취침 시각 전에 나온 기사(전날 수집 마감 뒤 ~ 취침 시각)를 묶은 것. 요약에는 넣지 않고 이 비교에만 쓴다. 사용자가 깨어 있을 때 나온 소식이라, 밤에 같은 사건이 나오면 후속으로 표시해야 한다. 마감 처리 때 요약 묶음보다 먼저 만들어 등록한다(9절).
  2. **지난 요약 묶음**: 같은 표시 시각으로 만든 지난 요약의 묶음. 방식을 바꾸기 전(11-16)의 1차·2차 요약 묶음도 넣는다.
- **지난 묶음 기록**: `source/stories/story_clusters.csv`와 `story_vecs.npz`(같은 줄 순서의 중심 벡터). 칸: 사건 번호(`story_id`), 보고 날짜, 이름(`낮·저녁` 또는 `08:00 요약`, 옛 기록은 `1차`/`2차`), 종류(`비교용 낮·저녁`/`요약`), 표시 시각, 묶음 번호, 등록 시각, 가장 이른·늦은 발행 시각, 제목, 매체 수, 기사 수. 같은 보고 날짜·같은 이름은 다시 실행하면 지우고 다시 넣는다.
- 기준 0.75는 임시값이다(같은 날 묶기 기준과 같게 시작, `news_search.md` 결정할 것 5).

In [ ]:
# ── 이전 소식 확인: 묶음을 지난 묶음 기록과 비교해 이미 알려진 소식의 후속인지, 처음 나온 소식인지 ──
STORY_DIR = ROOT / 'source' / 'stories'          # 지난 묶음 기록(보고 날짜와 상관없이 하나)
STORY_TABLE = STORY_DIR / 'story_clusters.csv'   # 묶음마다 한 줄: 사건 번호, 보고 날짜, 이름, 종류, 표시 시각, 묶음 번호, 등록 시각, 발행 시각 범위, 제목, 매체 수, 기사 수
STORY_VECS = STORY_DIR / 'story_vecs.npz'        # 같은 줄 순서의 묶음 중심 벡터
STORY_COLS = ['story_id', 'report', 'label', 'kind', 'summary_time', 'cluster', 'time_utc', 'first_published_utc', 'last_published_utc',
              'title', 'n_outlets', 'n_articles']
STORY_DAYS = 2              # 비교 기간: 이번 묶음의 첫 기사 미국 동부 날짜에서 이만큼 전 날짜의 00:00부터
STORY_THRESHOLD = 0.75      # 이 이상이면 같은 사건으로 본다(임시값: 같은 날 묶기 기준과 같게 시작)
KIND_DAY = '비교용 낮·저녁'  # 요약에 넣지 않고 이전 소식 확인에만 쓰는 낮·저녁 기사 묶음
KIND_SUMMARY = '요약'
STATUS_NEW, STATUS_FOLLOW = '처음 나온 소식', '이전 소식의 후속'


def load_stories():
    """지난 묶음 기록 → (표, 중심 벡터 행렬)."""
    if not STORY_TABLE.exists():
        return pd.DataFrame(columns=STORY_COLS), np.zeros((0, 3072), 'float32')
    S = pd.read_csv(STORY_TABLE, dtype={'report': str, 'summary_time': str}).fillna({'summary_time': '', 'kind': KIND_SUMMARY})
    return S, np.load(STORY_VECS)['vecs']


def compare_start_utc(first_published_utc):
    """비교 기간의 시작: 첫 기사의 미국 동부 날짜에서 STORY_DAYS일 전 00:00(미국 동부) → UTC."""
    d = pd.Timestamp(first_published_utc).tz_convert(ET).normalize() - pd.Timedelta(days=STORY_DAYS)
    return d.tz_convert('UTC')


def link_stories(C, centers, nid, label, kind, now=None):
    """묶음 표 C(cluster, first_published_utc, last_published_utc, title, n_outlets, n_articles 칸)에 이전 소식 확인 결과를 붙이고 지난 묶음 기록에 더한다.

    붙이는 칸: story_id(사건 번호), story(처음 나온 소식/이전 소식의 후속), prev(같은 사건의 지난 묶음: 이름, 발행 시각, 제목).
    비교 대상: 지난 묶음 기록 중 ① 비교용 낮·저녁 묶음 ② 같은 표시 시각의 요약 묶음(방식을 바꾸기 전 1차·2차 포함)이면서,
    가장 늦은 발행 시각이 비교 기간(이번 묶음 첫 기사의 미국 동부 날짜 2일 전 00:00 뒤) 안인 것.
    같은 보고 날짜·같은 이름으로 이미 등록된 것은 지우고 다시 넣는다(다시 실행해도 겹치지 않게).
    """
    now = now or now_utc()
    S, V = load_stories()
    keep = ~((S['report'] == nid) & (S['label'] == label))
    S, V = S[keep.values].reset_index(drop=True), V[keep.values]
    usable = (S['kind'] == KIND_DAY) | S['summary_time'].isin(['', SUMMARY_TIME])   # 빈 표시 시각 = 방식을 바꾸기 전 1차·2차
    last_pub = pd.to_datetime(S['last_published_utc'], utc=True, format='ISO8601')
    next_id = int(S['story_id'].max()) + 1 if len(S) else 1
    ids, status, prev, rows, vecs = [], [], [], [], []
    for r in C.itertuples():
        v = centers[r.cluster] / np.linalg.norm(centers[r.cluster])
        cand = np.where((usable & (last_pub >= compare_start_utc(r.first_published_utc))).values)[0]
        sims = V[cand] @ v if len(cand) else np.array([])
        j = int(cand[sims.argmax()]) if len(sims) else -1
        if j >= 0 and sims.max() >= STORY_THRESHOLD:
            sid = int(S.at[j, 'story_id'])
            status.append(STATUS_FOLLOW)
            earlier = S.iloc[cand][S.iloc[cand]['story_id'] == sid].sort_values('last_published_utc')
            prev.append(' || '.join(f"[{lb}, 발행 {pd.Timestamp(fp).tz_convert(ET):%m-%d %H:%M}~{pd.Timestamp(lp).tz_convert(ET):%m-%d %H:%M} 미국 동부] {ti}"
                                    for lb, fp, lp, ti in zip(earlier['label'], earlier['first_published_utc'], earlier['last_published_utc'], earlier['title'])))
        else:
            sid, next_id = next_id, next_id + 1
            status.append(STATUS_NEW)
            prev.append('')
        ids.append(sid)
        rows.append(dict(story_id=sid, report=nid, label=label, kind=kind, summary_time='' if kind == KIND_DAY else SUMMARY_TIME,
                         cluster=int(r.cluster), time_utc=now.isoformat(), first_published_utc=r.first_published_utc,
                         last_published_utc=r.last_published_utc, title=r.title, n_outlets=int(r.n_outlets), n_articles=int(r.n_articles)))
        vecs.append(v.astype('float32'))
    C = C.assign(story_id=ids, story=status, prev=prev)
    STORY_DIR.mkdir(parents=True, exist_ok=True)
    S2 = pd.concat([S, pd.DataFrame(rows, columns=STORY_COLS)], ignore_index=True)[STORY_COLS]
    V2 = np.vstack([V] + ([np.stack(vecs)] if vecs else []))
    S2.to_csv(STORY_TABLE, index=False, encoding='utf-8-sig')
    np.savez(STORY_VECS, vecs=V2)
    return C


def cluster_centers(I):
    """기사 표(cluster 칸) → {묶음 번호: 중심 벡터}(임베딩 캐시에서)."""
    X = embed(I['title'].tolist(), run=False)
    return {c: X[idx].mean(0) for c, idx in I.reset_index(drop=True).groupby('cluster').indices.items()}

## 8. 프롬프트 입력

요약에 넣을 묶음마다 밤사이 프롬프트 요청 1건에 들어갈 내용을 JSON 한 줄로 만든다(`nightly_prompt_requirements.md` 2장). 팀원이 이 파일(`prompt_inputs.jsonl`)로 프롬프트를 바로 시험할 수 있다.

| 칸 | 내용 |
|---|---|
| `report`, `label`, `cluster`, `rank` | 보고 날짜, 이름(예: `08:00 요약`), 묶음 번호, 중요도 순위 |
| `night` | 표시 시각, 취침 시각, 수집 마감(한국 시간): 이 요약에 넣은 기사는 취침 시각 뒤에 발행되고 수집 마감 전에 받은 것 |
| `importance` | 서로 다른 매체 수, 기사 수, 매체 목록 |
| `story` | 이전 소식 확인 결과(7절): 사건 번호, `처음 나온 소식`/`이전 소식의 후속`, 같은 사건의 지난 묶음(이름, 발행 시각 범위, 제목) |
| `article` | 본문 기사: 제목, 매체, 발행 시각, 원문 주소, 본문(받지 못하면 빈 글자와 `body_ok: false`) |
| `other_titles` | 같은 묶음의 다른 제목(중심에 가까운 순 2개) |
| `companies` | 이 묶음을 고른 종목마다: 종목, 대표 제목·매체, 후보 제목, 찾은 검색어와 종류, 대표가 본문 기사와 같은지(`rep_is_body`), 그래프 연결 근거(6절) |

In [ ]:
# ── 프롬프트 입력: 묶음마다 요청 1건에 들어갈 내용(nightly_prompt_requirements.md 2장) ──
def build_prompt_inputs(nid, label, I, C, P, bodies, terms):
    """묶음마다 밤사이 프롬프트 입력 한 줄(JSON)을 만든다.

    I: 기사(묶음 번호 포함), C: 묶음 표(중요도, 이전 소식 확인 결과), P: 종목별 고른 묶음(대표·후보 제목), bodies: 묶음별 받은 본문(cluster, key, body, url)
    terms: 검색어 표(term, company, company_id, kind, node, theme)
    """
    art = I.set_index('key')
    body_of = {int(r.cluster): r for r in bodies.itertuples()}
    term_info = terms.drop_duplicates(['term', 'company']).set_index(['term', 'company'])
    _, bed, cut = night_times(nid)
    night = dict(summary_time=SUMMARY_TIME, bedtime_kst=bed.isoformat(), cutoff_kst=cut.isoformat())
    out = []
    for cl, g in P.groupby('cluster'):
        c = C.set_index('cluster').loc[cl]
        b = body_of.get(int(cl))
        bk = b.key if b is not None else g['rep_key'].iloc[0]
        a = art.loc[bk]
        companies = []
        for r in g.itertuples():
            ev = []
            for t in r.rep_terms.split(', '):
                if (t, r.company) in term_info.index:
                    ti = term_info.loc[(t, r.company)]
                    ev += graph_evidence(ti['company_id'], ti['node'], ti['kind'], ti['theme'])
            companies.append(dict(company=r.company, rep_title=r.rep_title, rep_outlet=r.rep_outlet,
                                  other_titles=[x for x in r.other_titles.split(' || ') if x], terms=r.rep_terms, kinds=r.rep_kind,
                                  rep_is_body=r.rep_key == bk, evidence=list(dict.fromkeys(ev))[:MAX_EVIDENCE]))
        out.append(dict(
            report=nid, label=label, cluster=int(cl), rank=int(c['rank']), night=night,
            importance=dict(n_outlets=int(c['n_outlets']), n_articles=int(c['n_articles']), outlets=c['outlets']),
            story=dict(story_id=int(c['story_id']), status=c['story'], earlier=[x for x in c['prev'].split(' || ') if x]) if 'story_id' in c else None,
            article=dict(title=a['title'], outlet=a['outlet'], published_utc=a['published_utc'],
                         url=b.url if b is not None else None, body=b.body if b is not None else '', body_ok=b is not None),
            other_titles=I[(I['cluster'] == cl) & (I['key'] != bk)].sort_values('center_sim', ascending=False)['title'].head(N_TITLES - 1).tolist(),
            companies=companies))
    return out

## 9. 수집 마감 뒤 처리 (프롬프트 입력까지)

수집 마감(표시 시각 30분 전, 기본 07:30)까지 받은 기사로 밤사이 프롬프트 직전까지 하고 단계별 시간을 잰다(`summary_run`). 보고 날짜 폴더의 기사를 **발행 시각**으로 둘로 나눈다(`night_times`).

| 나눈 기사 | 발행 시각 | 쓰는 곳 |
|---|---|---|
| 낮·저녁 기사 | 전날 수집 마감 ~ 취침 시각 | 비교용 낮·저녁 묶음으로 묶어 지난 묶음 기록에 등록만 한다(요약에 넣지 않음) |
| 요약에 넣을 밤 기사 | 취침 시각 뒤(수집 마감 전에 받은 것) | 아래 1~6단계 |

예(취침 00:00, 표시 08:00, 보고 날짜 11-17): 낮·저녁 기사는 11-16 07:30~11-17 00:00 발행, 요약에 넣을 밤 기사는 11-17 00:00 뒤에 발행되고 07:30 전에 받은 기사. 처음 받은 시각이 아니라 발행 시각으로 나누므로, 낮에 나온 기사를 밤 회차에서 처음 받아도 요약에 들어가지 않는다.

1. **비교용 낮·저녁 묶음 등록**: 낮·저녁 기사를 묶고(4절) 이전 소식 확인(7절)을 거쳐 지난 묶음 기록에 등록한다. 본문은 받지 않는다.
2. **묶기**: 요약에 넣을 밤 기사를 묶는다(4절). 중요도는 매체 수 → 기사 수.
3. **이전 소식 확인**(7절): 묶음마다 비교용 낮·저녁 묶음과 지난 요약 묶음에 비교해 `처음 나온 소식`/`이전 소식의 후속`.
4. **종목별 대표**(4절) → **종목마다 넘길 묶음 최대 10개**(`select_top`): ① 대표 제목에 그 종목의 직접적인 검색어(자기 회사, 해외 상대, 구체·넓은 제품, 브랜드)가 있는 묶음 ② 남는 자리는 중요도 순으로 채우되, **직접 검색어가 없는 묶음은 종목마다 3개까지만**(`MAX_INDIRECT_PER_COMPANY`). 중요도만 보면 그 종목만 다룬 작은 기사(매체 1곳)가 큰 사건에 밀리고, 10개를 다 채우면 검색어가 본문에만 스친 무관한 기사가 대부분이 된다(`news_search.md` 3.3).
5. **본문 받기**(5절, `pick_with_bodies`): 묶음마다 본문 기사 1건(가장 많은 종목이 대표로 고른 기사). 실패하면 같은 묶음의 다음 기사(중심에 가까운 순)로 최대 3건. **그래도 본문을 못 받은 묶음**(섹션 페이지, 보도자료, 영상 등)은 빼고, 그 종목의 다음 순위 묶음으로 빈자리를 채워 다시 받는다(최대 3번). 끝까지 못 채운 종목은 10개보다 적다.
6. **프롬프트 입력**(8절): `prompt_inputs.jsonl`.

결과는 `source/nightly/<보고 날짜>/summary_<표시 시각>/`(예: `summary_0800/`): `clusters.csv`(요약 묶음과 이전 소식 확인 결과), `day_clusters.csv`(비교용 낮·저녁 묶음), `items.csv`(요약에 넣을 밤 기사와 묶음 번호), `picks.csv`(종목별 고른 묶음과 대표·후보 제목), `bodies.csv`(본문 받기 결과, 시도마다, `round`는 몇 번째 받기인지), `bodies.jsonl`(성공한 본문), `prompt_inputs.jsonl`(프롬프트 입력), `timing.json`(단계별 걸린 시간과 수). 끝나면 텔레그램으로 걸린 시간을 알린다.

In [ ]:
# ── 수집 마감 뒤 처리: 낮·저녁 기사 → 비교용 묶음 등록 / 밤 기사 → 묶기 → 이전 소식 확인 → 종목별 10개 묶음 → 본문 받기 → 프롬프트 입력 ──
TOP_PER_COMPANY = 10         # 종목마다 본문 받기·밤사이 프롬프트로 넘길 묶음 수
MAX_REFILL_ROUNDS = 3        # 본문을 못 받은 묶음을 빼고 다음 순위로 채우는 최대 횟수(첫 받기 포함)
MAX_INDIRECT_PER_COMPANY = 3 # 종목마다, 대표 제목에 직접적인 검색어가 없는 묶음은 이만큼까지만 넣는다(이유는 news_search.md 3.3)
DIRECT_KINDS = {'자기 회사', '해외 상대', '구체 제품', '넓은 제품', '브랜드'}   # 제목에 있으면 먼저 넣는 검색어 종류


def limit_picks(R, k=TOP_PER_COMPANY, max_indirect=MAX_INDIRECT_PER_COMPANY):
    """줄 세운 표(select_top(k=None)) → 종목마다 최대 k개. 그중 대표 제목에 직접적인 검색어가 없는 묶음은 max_indirect개까지만."""
    nth = R.groupby(['company', 'direct']).cumcount()
    return R[R['direct'] | (nth < max_indirect)].groupby('company').head(k)


def select_top(S, C, k=TOP_PER_COMPANY):
    """종목마다 넘길 묶음: ① 대표 제목에 직접적인 검색어가 있는 묶음 ② 남는 자리는 중요도 순(매체 수 → 기사 수)으로 채우되,
    직접 검색어가 없는 묶음은 종목마다 MAX_INDIRECT_PER_COMPANY개까지만. 합쳐서 최대 k개.
    k=None이면 자르지 않고 종목마다 이 순서로 줄 세운 전체를 돌려준다(본문을 못 받은 묶음 대신 다음 순위를 채울 때)."""
    S = S.merge(C[['cluster', 'rank', 'n_outlets', 'n_articles']], on='cluster')
    S['direct'] = S['rep_in_title'] & S['rep_kind'].map(lambda s: bool(DIRECT_KINDS & set(s.split(', '))))
    S = S.sort_values(['company', 'direct', 'rank'], ascending=[True, False, True]).reset_index(drop=True)
    return limit_picks(S, k).reset_index(drop=True) if k else S


def split_night_items(nid):
    """보고 날짜 폴더의 기사(거르기 통과, 수집 마감 전에 받은 것)를 발행 시각으로 나눈다.
    반환: (낮·저녁 기사: 전날 수집 마감 ≤ 발행 < 취침 시각, 요약에 넣을 밤 기사: 발행 ≥ 취침 시각, 검색어–기사 연결)."""
    prev_cut, bed, cut = night_times(nid)
    items, hits, _ = load_night(nid)
    kept = items[items['drop'] == ''].copy()
    seen = pd.to_datetime(kept['first_seen_utc'], utc=True, format='ISO8601')
    pub = pd.to_datetime(kept['published_utc'], utc=True, format='ISO8601')
    kept, pub = kept[seen <= cut], pub[seen <= cut]
    day = kept[(pub >= prev_cut) & (pub < bed)].reset_index(drop=True)
    night = kept[pub >= bed].reset_index(drop=True)
    return day, night, hits


def fetch_cluster_bodies(B, I, log=print):
    """묶음별 본문 받기: 본문 기사부터, 실패하면 같은 묶음의 다음 기사(중심에 가까운 순)로 최대 MAX_BODY_TRIES건.
    반환: (성공 본문 표, 시도 기록, 걸린 초)."""
    order = {}
    for c, b in zip(B['cluster'], B['body_key']):
        rest = [k for k in I[I['cluster'] == c].sort_values('center_sim', ascending=False)['key'] if k != b]
        order[c] = list(dict.fromkeys([b] + rest))[:MAX_BODY_TRIES]
    got, tries, secs = {}, [], dict(resolve=0.0, fetch=0.0, extract=0.0)
    for attempt in range(MAX_BODY_TRIES):
        want = {c: ks[attempt] for c, ks in order.items() if c not in got and attempt < len(ks)}
        if not want:
            break
        R = get_bodies(I.drop_duplicates('key').set_index('key').loc[list(want.values())].reset_index(), log=log)
        for k, v in R.attrs['seconds'].items():
            secs[k] += v
        R['cluster'] = R['key'].map({k: c for c, k in want.items()})
        R['attempt'] = attempt + 1
        tries.append(R)
        got.update({int(r.cluster): r for r in R.itertuples() if r.status == 'SUCCESS'})
    TRY = pd.concat(tries, ignore_index=True) if tries else pd.DataFrame(columns=['key', 'status', 'cluster', 'body', 'final_url'])
    OK = pd.DataFrame([dict(cluster=c, key=r.key, body=r.body, url=r.final_url) for c, r in got.items()],
                      columns=['cluster', 'key', 'body', 'url'])
    return OK, TRY, secs


def pick_with_bodies(S, C, I, log=print):
    """종목마다 묶음 TOP_PER_COMPANY개를 고르되, 본문을 받은 묶음만 남긴다.

    1. 종목마다 순위(select_top)대로 고르고(최대 10개, 직접 검색어가 없는 묶음은 3개까지), 아직 받아 보지 않은 묶음의 본문을 받는다(묶음마다 최대 MAX_BODY_TRIES건).
    2. 본문을 끝내 못 받은 묶음(섹션 페이지, 보도자료, 영상 등)은 빼고, 빈자리를 그 종목의 다음 순위 묶음으로 채워 1을 되풀이한다.
    3. 새로 받을 묶음이 없거나 MAX_REFILL_ROUNDS번 했으면 멈춘다. 끝까지 못 채운 종목은 10개보다 적다.
    반환: (종목별 고른 묶음 P, 묶음별 본문 기사 B, 성공 본문 OK, 시도 기록 TRY, 걸린 초, 받기 횟수, 뺀 묶음 수)
    """
    ranked = select_top(S, C, k=None)
    ok, failed, bodies, tries, Bs = set(), set(), [], [], []
    secs = dict(resolve=0.0, fetch=0.0, extract=0.0)
    rounds = 0
    for rounds in range(1, MAX_REFILL_ROUNDS + 1):
        P = limit_picks(ranked[~ranked['cluster'].isin(failed)])
        need = set(P['cluster']) - ok - failed
        if not need:
            rounds -= 1
            break
        B = cluster_body_article(P[P['cluster'].isin(need)], I)
        OK, TRY, s = fetch_cluster_bodies(B, I, log=log)
        for k, v in s.items():
            secs[k] += v
        TRY['round'] = rounds
        bodies.append(OK); tries.append(TRY); Bs.append(B)
        ok |= set(OK['cluster'])
        failed |= need - set(OK['cluster'])
        if need - set(OK['cluster']):
            log(f'  본문을 못 받은 묶음 {len(need - set(OK["cluster"]))}개를 빼고 다음 순위로 채운다({rounds}번째 받기 뒤)')
    P = limit_picks(ranked[ranked['cluster'].isin(ok)]).reset_index(drop=True)
    OK = pd.concat(bodies, ignore_index=True) if bodies else pd.DataFrame(columns=['cluster', 'key', 'body', 'url'])
    TRY = pd.concat(tries, ignore_index=True) if tries else pd.DataFrame(columns=['key', 'status', 'cluster'])
    B = pd.concat(Bs, ignore_index=True) if Bs else pd.DataFrame(columns=['cluster', 'body_key', 'n_companies'])
    return P, B[B['cluster'].isin(set(P['cluster']))], OK[OK['cluster'].isin(set(P['cluster']))], TRY, secs, rounds, len(failed)


def summary_run(nid, terms, embed_run=EMBED_RUN, log=print):
    """보고 날짜 nid의 수집 마감 뒤 처리: 밤사이 프롬프트 직전까지 하고 단계별 걸린 시간을 남긴다.

    1. 낮·저녁 기사(전날 수집 마감 ~ 취침 시각 발행)를 묶어 비교용 낮·저녁 묶음으로 지난 묶음 기록에 등록한다(요약에 넣지 않음).
    2. 요약에 넣을 밤 기사(취침 시각 뒤 발행, 수집 마감 전에 받음)를 묶는다.
    3. 이전 소식 확인(7절) → 종목마다 10개 묶음과 본문 받기(본문을 못 받은 묶음은 빼고 다음 순위로 채움) → 프롬프트 입력(prompt_inputs.jsonl).
    결과: source/nightly/<보고 날짜>/summary_<표시 시각>/ (clusters.csv, day_clusters.csv, items.csv, picks.csv, bodies.csv, bodies.jsonl,
          prompt_inputs.jsonl, timing.json)
    """
    label = f'{SUMMARY_TIME} 요약'
    out = night_dir(nid) / f'summary_{CUTOFF_TAG}'
    out.mkdir(parents=True, exist_ok=True)
    prev_cut, bed, cut = night_times(nid)
    T = {'start_kst': dt.datetime.now(KST).isoformat(), 'summary_time': SUMMARY_TIME, 'bedtime_kst': bed.isoformat(), 'cutoff_kst': cut.isoformat()}
    t = time.time()
    day, night, hits = split_night_items(nid)
    log(f'[{nid} {label}] 낮·저녁 기사 {len(day)}건(발행 {prev_cut:%m-%d %H:%M}~{bed:%m-%d %H:%M}), '
        f'요약에 넣을 밤 기사 {len(night)}건(발행 {bed:%m-%d %H:%M} 뒤, {cut:%m-%d %H:%M}까지 받음)')
    # 1. 비교용 낮·저녁 묶음: 묶어서 지난 묶음 기록에 등록만 한다
    if len(day):
        Id, Cd = build_clusters(day, run=embed_run, log=log)
        Cd = link_stories(Cd, cluster_centers(Id), nid, '낮·저녁', KIND_DAY)
    else:
        Cd = pd.DataFrame(columns=['cluster', 'title', 'story_id', 'story'])
    T['day_s'] = round(time.time() - t, 1); t = time.time()
    if not len(night):
        raise RuntimeError(f'요약에 넣을 밤 기사가 없다(발행 {bed:%m-%d %H:%M} 뒤, {cut:%m-%d %H:%M}까지 받은 것)')
    # 2. 요약에 넣을 밤 기사 묶기
    I, C = build_clusters(night, run=embed_run, log=log)
    T['cluster_s'] = round(time.time() - t, 1); t = time.time()
    # 3. 이전 소식 확인: 비교용 낮·저녁 묶음과 같은 표시 시각의 지난 요약 묶음에 비교
    C = link_stories(C, cluster_centers(I), nid, label, KIND_SUMMARY)
    T['story_s'] = round(time.time() - t, 1); t = time.time()
    # 4. 종목별 대표
    S = pick_representatives(I, hits, terms)
    T['pick_s'] = round(time.time() - t, 1); t = time.time()
    # 5. 종목마다 10개 묶음과 본문 받기: 본문을 못 받은 묶음은 빼고 다음 순위로 채운다
    P, B, OK, TRY, secs, n_rounds, n_failed = pick_with_bodies(S, C, I, log=log)
    T.update({f'{k}_s': round(v, 1) for k, v in secs.items()}, body_rounds=n_rounds, body_failed_clusters=n_failed)
    T['bodies_s'] = round(time.time() - t, 1); t = time.time()
    # 6. 프롬프트 입력
    inputs = build_prompt_inputs(nid, label, I, C, P, OK, terms)
    T['inputs_s'] = round(time.time() - t, 1)
    T['end_kst'] = dt.datetime.now(KST).isoformat()
    T.update(day_articles=len(day), day_clusters=len(Cd), day_follow=int((Cd['story'] == STATUS_FOLLOW).sum()),
             articles=len(night), clusters=len(C), new_stories=int((C['story'] == STATUS_NEW).sum()),
             linked_stories=int((C['story'] == STATUS_FOLLOW).sum()), company_picks=len(P), body_clusters=len(B) + n_failed, body_ok=len(OK),
             body_status=TRY['status'].value_counts().to_dict() if len(TRY) else {})
    # 저장
    C.to_csv(out / 'clusters.csv', index=False, encoding='utf-8-sig')
    Cd.to_csv(out / 'day_clusters.csv', index=False, encoding='utf-8-sig')
    I.to_csv(out / 'items.csv', index=False, encoding='utf-8-sig')
    P.to_csv(out / 'picks.csv', index=False, encoding='utf-8-sig')
    TRY.drop(columns=['body'], errors='ignore').to_csv(out / 'bodies.csv', index=False, encoding='utf-8-sig')
    (out / 'bodies.jsonl').write_text(''.join(json.dumps(dict(cluster=int(r.cluster), key=r.key, url=r.url, body=r.body), ensure_ascii=False) + '\n'
                                              for r in OK.itertuples()), encoding='utf-8')
    (out / 'prompt_inputs.jsonl').write_text(''.join(json.dumps(x, ensure_ascii=False) + '\n' for x in inputs), encoding='utf-8')
    (out / 'timing.json').write_text(json.dumps(T, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
    total = (dt.datetime.fromisoformat(T['end_kst']) - dt.datetime.fromisoformat(T['start_kst'])).total_seconds() / 60
    notify(f"[news_search] {nid} {label} 처리(프롬프트 입력까지) {total:.1f}분: 비교용 낮·저녁 묶음 {len(Cd)}개 등록 / "
           f"밤 기사 {len(night)} → 묶음 {len(C)}(처음 나온 소식 {T['new_stories']}, 이전 소식의 후속 {T['linked_stories']}) → "
           f"본문을 받은 묶음 {len(OK)}개(못 받아 뺀 묶음 {n_failed}개, 받기 {n_rounds}번) / 원문 주소 {secs['resolve']:.0f}초, 원문 받기 {secs['fetch']:.0f}초")
    return T

## 10. 대기하며 돌리기

이 칸을 켜 두면(`WATCH_RUN = True`) **30초마다** 처리할 수집 마감이 있는지 확인해 9절 `summary_run`을 돌린다.

- **언제 처리하나**: 수집 노트북이 남기는 수집 마감 표시 파일(`cutoff_<표시 시각>.json`, 예: `cutoff_0800.json`)이 생기면 바로 처리한다. 수집이 실제로 끝난 것을 기준으로 하므로 마지막 회차가 조금 늦게 끝나도 덜 받은 채로 처리하지 않는다.
- **표시 파일이 안 생길 때**: 수집 마감 시각에서 `NO_MARKER_WAIT_MIN`분(10분)이 지나도 표시 파일이 없으면(수집이 꺼졌을 때) 받아 둔 데까지로 처리한다.
- **이미 한 것은 건너뛴다**: 결과(`summary_<표시 시각>/timing.json`)나 실패 기록(`error.txt`)이 있으면 하지 않는다. 수집 마감 뒤 `WATCH_HOURS`시간(6시간) 안의 마감만 본다(그보다 늦게 켰으면 11절로 직접 돌린다).
- 처리가 실패하면 알리고 `summary_<표시 시각>/error.txt`를 남겨 같은 실패를 되풀이하지 않는다(고친 뒤 11절로 다시 돌리거나 error.txt를 지우면 다시 처리한다).
- 수집 노트북(`news_collect.ipynb` 11절)과 함께 켜 둔다. 멈추려면 커널을 중지한다.

In [ ]:
# ── 대기하며 돌리기: 수집 마감 표시 파일이 생기면 그 보고 날짜를 처리 ──
WATCH_RUN = False            # True여야 돌린다(멈출 때까지 계속)
CHECK_SEC = 30               # 처리할 마감이 있는지 확인하는 간격(초)
NO_MARKER_WAIT_MIN = 10      # 수집 마감 뒤 이만큼(분) 지나도 표시 파일이 없으면(수집이 꺼졌을 때) 받아 둔 데까지로 처리한다
WATCH_HOURS = 6              # 수집 마감 뒤 이 시간 안의 마감만 처리한다(그보다 늦게 켰으면 11절로 직접)


def pending_cutoffs(now=None):
    """처리할 수집 마감 [(보고 날짜, 표시 파일이 있나)]: 오늘·어제 수집 마감 중 지났고, 결과도 실패 기록도 없는 것."""
    now = now or dt.datetime.now(KST)
    out = []
    for k in (1, 0):
        cut = (now - dt.timedelta(days=k)).replace(hour=CUTOFF_KST[0], minute=CUTOFF_KST[1], second=0, microsecond=0)
        if not (cut <= now <= cut + dt.timedelta(hours=WATCH_HOURS)):
            continue
        nid = cut.strftime('%Y%m%d')
        done = NIGHTLY_DIR / nid / f'summary_{CUTOFF_TAG}'
        if (done / 'timing.json').exists() or (done / 'error.txt').exists():
            continue
        marker = (NIGHTLY_DIR / nid / f'cutoff_{CUTOFF_TAG}.json').exists()
        if marker or now >= cut + dt.timedelta(minutes=NO_MARKER_WAIT_MIN):
            out.append((nid, marker))
    return out


def run_waiting(check_sec=CHECK_SEC):
    """멈출 때까지: check_sec초마다 처리할 수집 마감을 찾아 처리한다(임베딩 요청 허용)."""
    notify(f'[news_search] 처리 대기 시작: 표시 시각 {SUMMARY_TIME}, 수집 마감 {CUTOFF_KST[0]:02d}:{CUTOFF_KST[1]:02d}, 취침 시각 {BEDTIME}')
    while True:
        for nid, marker in pending_cutoffs():
            if not marker:
                notify(f'[news_search] {nid}: 수집 마감 표시 파일이 {NO_MARKER_WAIT_MIN}분 넘게 없어 받아 둔 데까지로 처리한다(수집이 멈췄는지 확인).')
            try:
                summary_run(nid, TERMS, embed_run=True)
            except Exception as e:                     # 실패를 남기고 알린 뒤 다음으로(같은 실패를 되풀이하지 않게)
                d = night_dir(nid) / f'summary_{CUTOFF_TAG}'
                d.mkdir(parents=True, exist_ok=True)
                (d / 'error.txt').write_text(f'{dt.datetime.now(KST).isoformat()} {type(e).__name__}: {e}', encoding='utf-8')
                notify(f'[news_search] {nid} {SUMMARY_TIME} 요약 처리 실패: {type(e).__name__}: {str(e)[:300]}')
        time.sleep(check_sec)


print('지금 처리할 수집 마감:', pending_cutoffs() or '없음')
if WATCH_RUN:
    run_waiting()
else:
    print('WATCH_RUN = False: 돌리지 않았다.')

## 11. 한 보고 날짜 골라 돌리기

지난 보고 날짜를 골라 수집 마감 뒤 처리(9절)를 다시 돌린다(기준값을 바꿔 보거나, 실패한 처리를 고친 뒤 다시 할 때). 지금 고른 표시 시각·취침 시각(1절)으로 돌린다. 결과 파일은 덮어쓰고, 지난 묶음 기록에서 같은 보고 날짜의 `낮·저녁`·`<표시 시각> 요약` 줄은 지우고 다시 넣는다. 방식을 바꾸기 전(11-16까지)의 보고 날짜 폴더는 날짜 경계가 06:30이라 이 처리에 맞지 않는다.

In [ ]:
# ── 한 보고 날짜 골라 돌리기 ──
PROCESS_RUN = False          # True여야 돌린다(임베딩·원문 요청이 나간다)
NID = None                   # 보고 날짜(예: '20261117'). None이면 수집 마감 표시 파일이 있는 가장 최근 보고 날짜
if PROCESS_RUN:
    nid = NID or sorted(p.parent.name for p in NIGHTLY_DIR.glob(f'*/cutoff_{CUTOFF_TAG}.json'))[-1]
    summary_run(nid, TERMS, embed_run=True)
else:
    print('PROCESS_RUN = False: 돌리지 않았다.')

## 12. 결과 보기

보고 날짜 하나의 수집 결과와 수집 마감 뒤 처리 결과를 정리해 `source/nightly/<보고 날짜>/report.md`로 저장한다(요청 없음, 저장된 파일만 읽는다). 언제든 다시 돌릴 수 있다.

| 장 | 내용 | 무엇을 판단하나 |
|---|---|---|
| 1 수집 | 요청 수·성공·막힘(429·503·연결 끊김), 요청 간격, 수집 마감을 지켰나(마감 회차의 마지막 요청 ≤ 수집 마감), 시간대별 요청 수, 회차, 100건 한도에 걸린 검색어 | 요청 한도(밤 시간당 약 550건이 버티나), 마감 지키기, 등급을 올려야 할 검색어 |
| 2 검색어별 새 기사 수 | 검색어마다 처음 받은 기사 수를 시간대별로(총수, 시간당 최대·평균) | 등급 간격(드문 등급에서 시작해 50건 넘으면 올리기의 기준) |
| 3 수집 마감 뒤 처리 | 걸린 시간(단계별)과 표시 시각까지 여유, 낮·저녁 기사와 비교용 묶음 수, 요약에 넣을 밤 기사 → 묶음 → 종목별 고른 묶음 → 본문 성공률(본문을 못 받아 빼고 다음 순위로 채운 묶음 수, 받기 횟수), 이전 소식 확인 결과(처음 나온 소식/이전 소식의 후속)와 후속으로 본 예, 본문 판정, 중요도 상위 묶음, 종목별 고른 묶음과 대표 제목·검색어, 프롬프트 입력 예시 | 수집 마감 시각이 맞나, 묶기 기준 0.75·종목당 10개가 맞나, 이전 소식 확인이 맞게 되나, 본문 받기 성공률 |

방식을 바꾸기 전(11-16까지) 보고 날짜의 1차·2차 처리 결과는 그때 만든 `report.md`에 있다.

In [ ]:
# ── 결과 보기: 보고 날짜 하나의 수집 결과와 수집 마감 뒤 처리 결과를 정리해 report.md로 저장(요청 없음) ──
REPORT_SHOW = 5              # 표마다 보여 줄 줄 수


def _md_table(df, cols=None, n=None):
    """표 → 마크다운 표(글자 수가 길면 자른다)."""
    if df is None or len(df) == 0:
        return '(없음)\n'
    d = df[cols] if cols else df
    d = d.head(n) if n else d
    cut = lambda v: (str(v)[:90] + '…') if isinstance(v, str) and len(str(v)) > 90 else v
    head = '| ' + ' | '.join(map(str, d.columns)) + ' |\n|' + '---|' * len(d.columns) + '\n'
    return head + ''.join('| ' + ' | '.join(str(cut(v)).replace('|', '/') for v in r) + ' |\n' for r in d.itertuples(index=False))


def collection_summary(nid):
    """수집 결과: 요청 수·성공률·막힘, 시간대별 요청 수, 100건 한도, 수집 마감을 지켰는지."""
    items, hits, flog = load_night(nid)
    out = {}
    if not len(flog):
        return out
    f = flog.copy()
    f['t'] = pd.to_datetime(f['fetched_utc'], utc=True).dt.tz_convert(KST)
    st = f['status'].astype(str)
    out['요청'] = dict(전체=len(f), 성공=int((st == '200').sum()), 막힘_429_503=int(st.isin(['429', '503']).sum()),
                     연결끊김=int((st == '연결 끊김').sum()), 기타실패=int((~st.isin(['200', '429', '503', '연결 끊김'])).sum()),
                     회차=f['round'].nunique(), 처음=f['t'].min().strftime('%m-%d %H:%M'), 마지막=f['t'].max().strftime('%m-%d %H:%M'))
    gaps = f['t'].sort_values().diff().dt.total_seconds()
    out['요청 간격(초)'] = dict(중앙값=round(float(gaps.median()), 1), 쉰_시간_1분이상=int((gaps > 60).sum()))
    by_hour = f.groupby(f['t'].dt.strftime('%m-%d %H시')).agg(요청=('status', 'size'), 성공=('status', lambda s: int((s.astype(str) == '200').sum())),
                                                            한도100=('capped', lambda s: int(s.astype(bool).sum())))
    out['시간대별'] = by_hour.reset_index().rename(columns={'t': '시간대'})
    cap = f[f['capped'].astype(bool)]
    out['한도에 걸린 검색어'] = cap.groupby(['term', 'group']).size().rename('횟수').reset_index().sort_values('횟수', ascending=False)
    rounds = f.groupby(['round', 'label']).agg(시작=('t', 'min'), 끝=('t', 'max'), 요청=('status', 'size')).reset_index()
    rounds['시작'], rounds['끝'] = rounds['시작'].dt.strftime('%m-%d %H:%M'), rounds['끝'].dt.strftime('%H:%M:%S')
    out['회차'] = rounds
    mk = NIGHTLY_DIR / nid / f'cutoff_{CUTOFF_TAG}.json'   # 수집 마감을 지켰나: 마감 회차의 마지막 요청 ≤ 수집 마감
    if mk.exists():
        info = json.loads(mk.read_text(encoding='utf-8'))
        dl = dt.datetime.fromisoformat(info['deadline_kst'])
        last = f.loc[f['label'] == '수집 마감', 't'].max()
        out['수집 마감'] = dict(표시_시각=info.get('summary_time', SUMMARY_TIME), 마감=f'{dl:%H:%M}',
                              마지막_요청=f'{last:%H:%M:%S}' if pd.notna(last) else '-',
                              지킴=bool(pd.notna(last) and last <= dl + dt.timedelta(seconds=40)),
                              표시_파일_시각=dt.datetime.fromisoformat(info['done_kst']).strftime('%H:%M:%S'), 기사=info['articles'])
    return out


def term_volume(nid):
    """검색어별 새 기사 수(처음 받은 시각 기준)를 시간대별로: 등급 간격을 정하는 자료. 반환: 검색어 × 시간 표와 검색어별 요약."""
    items, hits, _ = load_night(nid)
    if not len(hits):
        return None, None
    first = hits.merge(items[['key', 'first_seen_utc', 'drop']], on='key')
    first = first[first['drop'] == ''].sort_values('first_seen_utc').drop_duplicates(['key', 'term'])
    first['시'] = pd.to_datetime(first['first_seen_utc'], utc=True).dt.tz_convert(KST).dt.strftime('%H')
    pv = first.pivot_table(index='term', columns='시', values='key', aggfunc='count', fill_value=0)
    summ = pd.DataFrame({'새기사': pv.sum(1), '시간당_최대': pv.max(1), '시간당_평균': pv.mean(1).round(1)})
    summ = summ.reindex(sorted(set(TERMS['term']) | set(summ.index)), fill_value=0)   # 새 기사가 없던 검색어도 넣는다(0건)
    return pv, summ.sort_values('새기사', ascending=False)


def processing_summary(nid):
    """수집 마감 뒤 처리 결과: 걸린 시간, 단계별 수, 이전 소식 확인 결과, 종목별 고른 묶음, 본문 상태, 프롬프트 입력."""
    d = NIGHTLY_DIR / nid / f'summary_{CUTOFF_TAG}'
    if not (d / 'timing.json').exists():
        return None
    T = json.loads((d / 'timing.json').read_text(encoding='utf-8'))
    C = pd.read_csv(d / 'clusters.csv')
    P = pd.read_csv(d / 'picks.csv')
    B = pd.read_csv(d / 'bodies.csv') if (d / 'bodies.csv').exists() else pd.DataFrame()
    X = [json.loads(l) for l in (d / 'prompt_inputs.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
    start, end = dt.datetime.fromisoformat(T['start_kst']), dt.datetime.fromisoformat(T['end_kst'])
    show_at = dt.datetime.fromisoformat(T['cutoff_kst']) + dt.timedelta(minutes=PREP_MIN)
    return dict(
        T=T, C=C, P=P, B=B, X=X,
        시간=dict(시작=f'{start:%H:%M:%S}', 끝=f'{end:%H:%M:%S}', 걸린_분=round((end - start).total_seconds() / 60, 1),
                 표시_시각까지_여유_분=round((show_at - end).total_seconds() / 60, 1), 낮저녁_묶음_초=T.get('day_s'),
                 묶기_초=T.get('cluster_s'), 이전소식확인_초=T.get('story_s'), 고르기_초=T.get('pick_s'), 원문주소_초=T.get('resolve_s'),
                 원문받기_초=T.get('fetch_s'), 본문추출_초=T.get('extract_s'), 입력만들기_초=T.get('inputs_s')),
        범위=dict(표시_시각=T['summary_time'], 취침_시각=dt.datetime.fromisoformat(T['bedtime_kst']).strftime('%m-%d %H:%M'),
                 수집_마감=dt.datetime.fromisoformat(T['cutoff_kst']).strftime('%m-%d %H:%M'),
                 낮저녁_기사=T['day_articles'], 비교용_낮저녁_묶음=T['day_clusters'], 그중_이전_소식의_후속=T['day_follow']),
        수=dict(요약에_넣을_밤_기사=T['articles'], 묶음=T['clusters'], 처음_나온_소식=T['new_stories'], 이전_소식의_후속=T['linked_stories'],
               종목별_고른_묶음=T['company_picks'], 본문_받을_묶음=T['body_clusters'], 본문_성공=T['body_ok'],
               본문_성공률=f"{T['body_ok'] / max(T['body_clusters'], 1):.0%}", 못_받아_뺀_묶음=T.get('body_failed_clusters', 0),
               받기_횟수=T.get('body_rounds', 1), 프롬프트_입력=len(X),
               본문없는_입력=sum(not x['article']['body_ok'] for x in X)),
        본문_판정=T.get('body_status', {}))


def night_report(nid, save=True):
    """보고 날짜 하나의 결과를 마크다운으로 정리한다(source/nightly/<보고 날짜>/report.md)."""
    L = [f'# 보고 날짜 {nid} 결과\n', f'만든 시각: {dt.datetime.now(KST):%Y-%m-%d %H:%M} (KST)\n']
    cs = collection_summary(nid)
    L.append('## 1. 수집\n')
    if not cs:
        L.append('(수집 기록 없음)\n')
    else:
        L.append(_md_table(pd.DataFrame([cs['요청']])))
        L.append(f"\n요청 간격 중앙값 {cs['요청 간격(초)']['중앙값']}초, 1분 넘게 쉰 곳 {cs['요청 간격(초)']['쉰_시간_1분이상']}번\n")
        if '수집 마감' in cs:
            L.append("\n- **수집 마감**: " + ', '.join(f'{k} {v}' for k, v in cs['수집 마감'].items()) + '\n')
        L.append('\n**시간대별 요청**(요청 한도 시험)\n\n' + _md_table(cs['시간대별']))
        L.append('\n**회차**\n\n' + _md_table(cs['회차'], ['label', '시작', '끝', '요청']))
        L.append('\n**100건 한도에 걸린 검색어**(등급을 올려야 할 것)\n\n' + _md_table(cs['한도에 걸린 검색어'], n=15))
    pv, summ = term_volume(nid)
    L.append('\n## 2. 검색어별 새 기사 수(등급 간격 자료)\n')
    if summ is None:
        L.append('(없음)\n')
    else:
        L.append(f"검색어 {len(summ)}개 중 새 기사가 있는 것 {int((summ['새기사'] > 0).sum())}개, 시간당 최대 20건 이상 {int((summ['시간당_최대'] >= 20).sum())}개\n\n")
        L.append(_md_table(summ.reset_index().rename(columns={'term': '검색어'}), n=20))
    ps = processing_summary(nid)
    L.append(f'\n## 3. 수집 마감 뒤 처리({SUMMARY_TIME} 요약, 프롬프트 입력까지)\n')
    if ps is None:
        err = NIGHTLY_DIR / nid / f'summary_{CUTOFF_TAG}' / 'error.txt'
        L.append(f"(결과 없음{': 실패 ' + err.read_text(encoding='utf-8')[:200] if err.exists() else ''})\n")
    else:
        L.append('**걸린 시간**\n\n' + _md_table(pd.DataFrame([ps['시간']])))
        L.append('\n**기사를 나눈 시각과 비교용 낮·저녁 묶음**\n\n' + _md_table(pd.DataFrame([ps['범위']])))
        L.append('\n**단계별 수**\n\n' + _md_table(pd.DataFrame([ps['수']])))
        L.append('\n본문 판정: ' + ', '.join(f'{k} {v}' for k, v in ps['본문_판정'].items()) + '\n')
        C = ps['C']
        L.append('\n**중요도 상위 묶음**\n\n' + _md_table(C.sort_values('rank'), ['rank', 'n_outlets', 'n_articles', 'story', 'title'], n=10))
        F = C[C['story'] == STATUS_FOLLOW].sort_values('rank')
        L.append('\n**이전 소식의 후속으로 본 묶음(앞 10개)과 같은 사건의 지난 묶음**\n\n' + _md_table(F, ['rank', 'title', 'prev'], n=10))
        P = ps['P']
        per = P.groupby('company').agg(고른_묶음=('cluster', 'size'), 제목에_직접검색어=('direct', 'sum')).reset_index()
        L.append('\n**종목별 고른 묶음 수**\n\n' + _md_table(per))
        L.append('\n**종목별 고른 묶음(앞 3개)**\n\n' + _md_table(P.groupby('company').head(3), ['company', 'rank', 'n_outlets', 'direct', 'rep_kind', 'rep_terms', 'rep_title']))
        if len(ps['X']):
            x = dict(ps['X'][0]); x['article'] = dict(x['article']); x['article']['body'] = x['article']['body'][:300] + '…'
            L.append('\n**프롬프트 입력 예시(1건, 본문 300자까지)**\n\n```json\n' + json.dumps(x, ensure_ascii=False, indent=1)[:3000] + '\n```\n')
    text = '\n'.join(L)
    if save:
        (night_dir(nid) / 'report.md').write_text(text, encoding='utf-8')
    return text


REPORT_NID = None            # 보고 날짜(예: '20261117'). None이면 가장 최근 보고 날짜
nid = REPORT_NID or sorted(p.name for p in NIGHTLY_DIR.iterdir() if p.is_dir() and p.name.isdigit())[-1]
rep = night_report(nid)
print(f'저장: source/nightly/{nid}/report.md ({len(rep):,}자)')
print(rep[:3000])

## 13. 확인: 3차 시험 기사로 묶기·대표 고르기 (요청 없음)

시험용 칸이다. 시험 자료는 `news_search_tests/source/`에 있다(없으면 드라이브 `news_tests` 묶음을 푼다). 3차 시험(11-12~14) 기사를 밤사이 결과 형식으로 바꿔 같은 함수로 돌려 본다. 임베딩은 시험 때 받아 둔 것(`news_search_tests/source/cluster_test/`)을 캐시에 옮겨 쓰므로 요청이 없다.

In [17]:
# ── 3차 시험 기사를 밤사이 형식(items, hits)으로 바꾸기 ──
TEST_DIR = PROJ / 'news_search_tests' / 'source'     # 시험 자료(검색 방식 시험 폴더)
K = pd.read_csv(TEST_DIR / 'rss_test_short' / 'kept_items.csv')
K = K[K['query'].isin(TERMS['term'])]
K['title'] = K['title'].map(clean_title)
K['outlet'] = K['domain'].map(base_outlet)
K['published_utc'] = pd.to_datetime(K['published'], utc=True).map(lambda t: t.isoformat())
K['key'] = [text_hash(o + '|' + t.lower()) for o, t in zip(K['outlet'], K['title'])]
T_ITEMS = K.drop_duplicates('key')[['key', 'title', 'source', 'domain', 'outlet', 'published_utc', 'link']].assign(drop='')
T_HITS = K[['key', 'query']].rename(columns={'query': 'term'}).drop_duplicates()

# 시험 때 받은 large 임베딩을 캐시에 옮긴다(이미 있으면 건너뜀)
cache = load_cache()
titles = pd.read_csv(TEST_DIR / 'cluster_test' / 'titles.csv')['text'].tolist()
vecs = np.load(TEST_DIR / 'cluster_test' / 'emb_text-embedding-3-large.npy')
added = 0
for t, v in zip(titles, vecs):
    k = text_hash(t)
    if k not in cache:
        cache[k] = v.astype('float32')
        added += 1
if added:
    save_cache(cache)
print(f'시험 기사 {len(T_ITEMS)}건, 검색어–기사 연결 {len(T_HITS)}건 / 캐시에 옮긴 임베딩 {added}개(캐시 {len(cache)}개)')

시험 기사 3997건, 검색어–기사 연결 5000건 / 캐시에 옮긴 임베딩 0개(캐시 6253개)


In [18]:
# ── 묶기(요청 없음) → 중요도 상위 묶음 ──
TI, TC = build_clusters(T_ITEMS, run=False)
print(f'묶음 {len(TC)}개, 2건 이상 {(TC["n_articles"] >= 2).sum()}개')
display(TC.head(15)[['rank', 'n_outlets', 'n_articles', 'title', 'outlets']])

묶음 3213개, 2건 이상 467개


,rank,n_outlets,n_articles,title,outlets
0,1,11,21,Nvidia names Colette Kress interim CEO after Huang resigns,"businessinsider.com, cbsnews.com, cnbc.com, cnn.com, finance.yahoo.com, fool.com, fort..."
1,2,8,25,Nvidia CEO Jensen Huang resigns after reported affairs,"businessinsider.com, cbsnews.com, cnbc.com, finance.yahoo.com, foxbusiness.com, nypost..."
2,3,8,16,Nvidia's new CEO search is on. Here are the top candidates to replace Jensen Huang.,"businessinsider.com, cbsnews.com, cnbc.com, finance.yahoo.com, fortune.com, foxbusines..."
3,4,7,21,Nvidia’s stock drops as board reportedly preps to oust CEO Jensen Huang,"cbsnews.com, cnbc.com, cnn.com, finance.yahoo.com, fool.com, foxbusiness.com, theguard..."
4,5,7,10,Trump calls for former Apple leader Tim Cook to replace Jensen Huang as Nvidia CEO,"businessinsider.com, cbsnews.com, cnbc.com, finance.yahoo.com, foxbusiness.com, nypost..."
5,6,6,9,Nvidia’s Jensen Huang also had alleged affair with subordinate as new report details C...,"businessinsider.com, cbsnews.com, cnbc.com, finance.yahoo.com, foxbusiness.com, nypost..."
6,7,6,7,Elon Musk says it would be ‘honor’ to hire ex-Nvidia CEO Jensen Huang,"businessinsider.com, cnbc.com, finance.yahoo.com, fortune.com, foxbusiness.com, nypost..."
7,8,5,10,Nvidia shares tumble after CEO Jensen Huang resigns,"businessinsider.com, cnbc.com, finance.yahoo.com, foxbusiness.com, nypost.com"
8,9,4,8,Nvidia stock slide wipes out more than $20B in value for Jensen Huang over 5 days,"cbsnews.com, finance.yahoo.com, fortune.com, foxbusiness.com"
9,10,4,6,Nvidia stock falls 8% after report board is preparing to oust Jensen Huang after secon...,"businessinsider.com, finance.yahoo.com, foxbusiness.com, nypost.com"


In [19]:
# ── 종목별 대표 기사와 후보 제목, 묶음별 본문 기사 ──
TS = pick_representatives(TI, T_HITS, TERMS)
TB = cluster_body_article(TS, TI)
TS = TS.merge(TC[['cluster', 'rank', 'n_outlets', 'n_articles']], on='cluster').sort_values(['company', 'rank'])
print(f'종목–묶음 {len(TS)}쌍 / 대표가 제목에 검색어를 가진 비율 {TS["rep_in_title"].mean():.0%}')
for company in ['SK하이닉스', '크래프톤', 'KB금융']:
    print(f'\n■ {company}: 중요도 상위 묶음 5개')
    for r in TS[TS['company'] == company].head(5).itertuples():
        print(f'  [{r.rank}위, 매체 {r.n_outlets}, 기사 {r.n_articles}] 대표: {r.rep_title[:80]} ({r.rep_outlet}, 검색어 {r.rep_terms}, {r.rep_kind})')
        print(f'      후보: {r.other_titles[:150]}')

종목–묶음 4344쌍 / 대표가 제목에 검색어를 가진 비율 18%

■ SK하이닉스: 중요도 상위 묶음 5개
  [1위, 매체 11, 기사 21] 대표: Nvidia names Colette Kress interim CEO after Huang resigns (usatoday.com, 검색어 "Nvidia", 해외 상대)
      후보: Who is Colette Kress? Nvidia's finance chief steps in as interim CEO after Huang exit || Nvidia’s Jensen Huang resigns, chipmaker names CFO Colette Kr
  [2위, 매체 8, 기사 25] 대표: Nvidia CEO Jensen Huang resigns after reported affairs (cbsnews.com, 검색어 "Nvidia", 해외 상대)
      후보: Nvidia CEO Jensen Huang resigns amid scandal over affair || Nvidia CEO Jensen Huang resigns in wake of affair, spy scandal
  [3위, 매체 8, 기사 16] 대표: Nvidia's new CEO search is on. Here are the top candidates to replace Jensen Hua (businessinsider.com, 검색어 "Nvidia", 해외 상대)
      후보: Nvidia's new CEO search is on. Here are the top candidates to replace Jensen Huang. || Nvidia’s next CEO: The outside contenders to succeed Jensen Hua
  [4위, 매체 7, 기사 21] 대표: Nvidia’s stock drops as board reportedly preps to oust CEO Jensen Huang (finan

In [20]:
# ── 한 묶음에 여러 종목이 걸릴 때: 종목마다 대표가 다른 예 ──
multi = TS.groupby('cluster').filter(lambda g: g['rep_key'].nunique() >= 2)
print(f'종목마다 대표가 다른 묶음 {multi["cluster"].nunique()}개 / 여러 종목이 걸린 묶음 {TS.groupby("cluster")["company"].nunique().gt(1).sum()}개')
body = TB.set_index('cluster')['body_key']
for c in multi.sort_values('rank')['cluster'].unique()[:4]:
    g = multi[multi['cluster'] == c]
    print(f"\n[묶음 {g['rank'].iloc[0]}위] 본문 기사: {TI.set_index('key').at[body[c], 'title'][:80]}")
    for r in g.itertuples():
        print(f'   {r.company:10} 대표: {r.rep_title[:70]} (검색어 {r.rep_terms})')

종목마다 대표가 다른 묶음 141개 / 여러 종목이 걸린 묶음 869개

[묶음 1위] 본문 기사: Nvidia CFO Colette Kress takes over as interim CEO—but the succession question r
   POSCO홀딩스   대표: Nvidia’s Jensen Huang resigns, chipmaker names CFO Colette Kress as in (검색어 "General Motors")
   SK하이닉스     대표: Nvidia names Colette Kress interim CEO after Huang resigns (검색어 "Nvidia")
   셀트리온       대표: Nvidia CFO Colette Kress takes over as interim CEO—but the succession  (검색어 "Novartis")
   크래프톤       대표: Who is Colette Kress? Nvidia's finance chief steps in as interim CEO a (검색어 "Microsoft")
   펩트론        대표: Nvidia CFO Colette Kress takes over as interim CEO—but the succession  (검색어 "Novartis")

[묶음 2위] 본문 기사: Nvidia CEO Jensen Huang resigns after reported affairs
   SK하이닉스     대표: Nvidia CEO Jensen Huang resigns after reported affairs (검색어 "Nvidia")
   크래프톤       대표: Nvidia CEO Jensen Huang resigns in wake of affair with woman suspected (검색어 "Microsoft")

[묶음 3위] 본문 기사: Nvidia's new CEO search is on. Here are the top candidates